In [ ]:
"""
data_loader.py — Единый загрузчик данных по BTC (1H и 6H бары).

Все функции принимают start_date в формате "YYYY-MM-DD".
Все функции возвращают DataFrame с индексом DatetimeIndex, tz-naive,
name="datetime", время — московское (MSK, UTC+3).

Источники данных:
    - Bybit API         : BTC OHLCV 1H/6H + taker_ratio + taker volumes
    - yfinance          : BTC + макро-индикаторы (1D → ffill 1H)
    - alternative.me    : Fear & Greed Index (1D → ffill 1H)
    - Bybit Futures API : Open Interest (1H)
    - Bybit Futures API : Funding Rate (8H → ffill 1H)
    - Bybit Futures API : Long/Short Ratio (1H)

Использование:
    from data_loader import load_all
    data = load_all(days=365)
"""

import logging
import time
import warnings
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# Константы


DOWNLOAD_DAYS = 720

_INTERVAL_1H_MS = 3_600_000
_INTERVAL_6H_MS = 21_600_000
_INTERVAL_1D_MS = 86_400_000

_FR_CHUNK_MS = 5_184_000_000 # ~60 дней
_LS_CHUNK_MS = 6_912_000_000 # ~80 дней


_MSK = timezone(timedelta(hours=3))


_FR_ENDPOINT = "https://api.bybit.com/v5/market/funding/history"

# Лаги публикации для разных типов источников (в часах MSK).
_DAILY_PUBLICATION_LAG_H = 24

_FUNDING_PUBLICATION_LAG_H = 1

_INTRABAR_PUBLICATION_LAG_H = 1



# HTTP-сессия с retry


_SESSION = None


def _get_session() -> requests.Session:
    """Возвращает переиспользуемую HTTP-сессию с retry (для Bybit)."""
    global _SESSION
    if _SESSION is None:
        session = requests.Session()
        retry = Retry(
            total=5,
            backoff_factor=2.0,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("https://", adapter)
        session.headers.update(
            {"User-Agent": "Mozilla/5.0 DataLoader/3.0"}
        )
        _SESSION = session
    return _SESSION


# Вспомогательные функции

def _to_dt(date_str: str) -> datetime:
    return datetime.strptime(date_str, "%Y-%m-%d")


def _to_ms(date_str: str) -> int:
    return int(_to_dt(date_str).timestamp() * 1000)


def _utc_to_msk(dt: datetime) -> datetime:
    """Конвертирует UTC datetime в MSK (UTC+3), tz-naive."""
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(_MSK).replace(tzinfo=None)


def _normalize_index(df: pd.DataFrame, name: str = "datetime") -> pd.DataFrame:
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df.index.name = name
    return df.sort_index()


def _ms_col_to_index(df: pd.DataFrame, col: str = "timestamp") -> pd.DataFrame:
    """Преобразует колонку с Unix-ms в DatetimeIndex (MSK, tz-naive)."""
    df = df.copy()
    ts = pd.to_datetime(df[col], unit="ms", utc=True)
    ts = ts.dt.tz_convert(_MSK).dt.tz_localize(None)
    df.index = ts.rename("datetime")
    df = df.drop(columns=[col])
    df = df[~df.index.duplicated(keep="last")]
    return df.sort_index()


def _end_date_default() -> str:
    """Завтра в MSK, формат YYYY-MM-DD."""
    return (datetime.now(_MSK) + timedelta(days=1)).strftime("%Y-%m-%d")

# Выравнивание источников на мастер-индекс

def _align_with_publication_lag(
    df: pd.DataFrame,
    master_index: pd.DatetimeIndex,
    lag_hours: int,
) -> pd.DataFrame:
    """
    Выравнивает df на master_index через merge_asof с обязательным
    сдвигом, имитирующим задержку публикации.

    Логика: значение источника со штампом T становится доступным только
    в баре T + lag_hours. Для каждого бара мастер-индекса берём последнее
    значение источника, чей штамп + lag <= штамп бара.

    Это математически эквивалентно: target_idx <- df.index + lag_hours,
    затем merge_asof(direction="backward") без tolerance.
    """
    if df.empty:
        return df

    df = df.copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df.index = df.index + pd.Timedelta(hours=lag_hours)
    df.index.name = "datetime"

    target = pd.DataFrame(index=master_index.sort_values())
    target.index.name = "datetime"

    merged = pd.merge_asof(
        target.reset_index(),
        df.reset_index(),
        on="datetime",
        direction="backward",
    )
    merged = merged.set_index("datetime")
    return merged.reindex(master_index)


def _align_daily(
    df: pd.DataFrame,
    master_index: pd.DatetimeIndex,
) -> pd.DataFrame:
    """Дневные источники: macro, fear & greed, bybit_daily."""
    return _align_with_publication_lag(
        df, master_index, _DAILY_PUBLICATION_LAG_H,
    )


def _align_funding(
    df: pd.DataFrame,
    master_index: pd.DatetimeIndex,
) -> pd.DataFrame:
    """Funding rate (8H): значение фиксируется в момент funding."""
    return _align_with_publication_lag(
        df, master_index, _FUNDING_PUBLICATION_LAG_H,
    )


def _align_intrabar(
    df: pd.DataFrame,
    master_index: pd.DatetimeIndex,
) -> pd.DataFrame:
    """OI, L/S Ratio (1H): значение бара T доступно в баре T+1."""
    return _align_with_publication_lag(
        df, master_index, _INTRABAR_PUBLICATION_LAG_H,
    )


# Bybit kline

def _bybit_paginate(endpoint: str, params: dict) -> list:
    """Итерирует по страницам Bybit API с cursor-пагинацией."""
    all_records = []
    cursor = None
    session = _get_session()
    params = params.copy()

    while True:
        if cursor:
            params["cursor"] = cursor
        try:
            resp = session.get(endpoint, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            if data["retCode"] != 0:
                logger.error("Bybit API ошибка: %s", data["retMsg"])
                break
            records = data["result"]["list"]
            next_cursor = data["result"].get("nextPageCursor", "")
            if not records:
                break
            all_records.extend(records)
            if not next_cursor or next_cursor == cursor:
                break
            cursor = next_cursor
            time.sleep(0.12)
        except Exception as exc:
            logger.error("Ошибка запроса %s: %s", endpoint, exc)
            break
    return all_records


_KLINE_LIMIT = 1000
_KLINE_SLEEP = 0.4
_KLINE_RATELIMIT_SLEEP = 5


def _bybit_kline(
    start_date: str,
    end_date: str,
    symbol: str,
    interval_str: str,
    interval_ms: int,
) -> pd.DataFrame:
    """Скачивает OHLCV свечи с Bybit API."""
    base_url = "https://api.bybit.com/v5/market/kline"
    start_ms = _to_ms(start_date)
    end_ms = _to_ms(end_date)
    all_rows = []
    cur_start = start_ms
    session = _get_session()

    while cur_start < end_ms:
        cur_end = min(cur_start + _KLINE_LIMIT * interval_ms, end_ms)
        try:
            resp = session.get(
                base_url,
                params={
                    "category": "linear",
                    "symbol": symbol,
                    "interval": interval_str,
                    "start": cur_start,
                    "end": cur_end,
                    "limit": _KLINE_LIMIT,
                },
                timeout=15,
            )
            resp.raise_for_status()
            data = resp.json()
            ret_code = data.get("retCode", -1)
            if ret_code == 10006:
                logger.warning(
                    "Bybit rate limit, пауза %d сек...",
                    _KLINE_RATELIMIT_SLEEP,
                )
                time.sleep(_KLINE_RATELIMIT_SLEEP)
                continue
            if ret_code != 0:
                logger.error("Bybit kline API: %s", data.get("retMsg"))
                break
            rows = data["result"]["list"]
            if rows:
                all_rows.extend(rows)
        except Exception as exc:
            logger.error("Ошибка запроса Bybit kline: %s", exc)
        cur_start = cur_end
        time.sleep(_KLINE_SLEEP)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(
        all_rows,
        columns=[
            "timestamp", "open", "high", "low",
            "close", "volume", "turnover",
        ],
    )
    df["timestamp"] = df["timestamp"].astype(int)
    for col in ["open", "high", "low", "close", "volume", "turnover"]:
        df[col] = df[col].astype(float)

    return _ms_col_to_index(df, col="timestamp")


# Загрузчики данных — свечи

def load_candles_1h(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """1H OHLCV свечи BTC с Bybit (MSK)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Bybit 1H candles | %s → %s", start_date, end_date)

    df = _bybit_kline(
        start_date, end_date,
        symbol=symbol,
        interval_str="60",
        interval_ms=_INTERVAL_1H_MS,
    )
    if df.empty:
        logger.warning("Bybit 1H: нет данных")
        return pd.DataFrame()

    hl = (df["high"] - df["low"]).replace(0, np.nan)
    df["taker_ratio"] = ((df["close"] - df["low"]) / hl).clip(0, 1)
    df["taker_buy_vol"] = df["turnover"] * df["taker_ratio"]
    df["taker_sell_vol"] = df["turnover"] * (1 - df["taker_ratio"])
    df["taker_vol_ratio"] = (
        df["taker_buy_vol"] / df["taker_sell_vol"].replace(0, np.nan)
    )

    logger.info(
        "Bybit 1H готово: %d свечей | %s — %s",
        len(df), df.index[0], df.index[-1],
    )
    return df


def load_candles_6h(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """6H OHLCV свечи BTC с Bybit (MSK)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Bybit 6H candles | %s → %s", start_date, end_date)

    df = _bybit_kline(
        start_date, end_date,
        symbol=symbol,
        interval_str="360",
        interval_ms=_INTERVAL_6H_MS,
    )
    if df.empty:
        logger.warning("Bybit 6H: нет данных")
        return pd.DataFrame()

    logger.info(
        "Bybit 6H готово: %d свечей | %s — %s",
        len(df), df.index[0], df.index[-1],
    )
    return df


def load_bybit_daily(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """Дневные OHLCV свечи BTC с Bybit (MSK)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Bybit Daily | %s → %s", start_date, end_date)

    df = _bybit_kline(
        start_date, end_date,
        symbol=symbol,
        interval_str="D",
        interval_ms=_INTERVAL_1D_MS,
    )
    if df.empty:
        logger.warning("Bybit Daily: нет данных")
        return pd.DataFrame()

    logger.info(
        "Bybit Daily готово: %d свечей | %s — %s",
        len(df), df.index[0].date(), df.index[-1].date(),
    )
    return df


# Макро-индикаторы


MACRO_TICKERS = {
    "BTC-USD": "Bitcoin",
    "SPY": "S&P 500 ETF",
    "DX-Y.NYB": "Dollar Index (DXY)",
    "^VIX": "CBOE VIX Index",
    "TLT": "20+ Year Treasury",
    "IEF": "7-10 Year Treasury",
    "GLD": "Gold ETF",
    "USO": "Oil ETF (WTI)",
    "^TNX": "10Y Treasury Yield",
    "^MOVE": "ICE BofA MOVE Index (Bond VIX)",
}


def _safe_yf_download(
    ticker: str,
    start: str,
    end: str,
    retries: int = 3,
    delay: float = 1.0,
) -> pd.DataFrame:
    """Скачивает 1D данные с yfinance с повторными попытками."""
    yf_logger = logging.getLogger("yfinance")
    yf_logger.setLevel(logging.CRITICAL)
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(
                ticker, start=start, end=end, interval="1d",
                progress=False, auto_adjust=True, threads=False,
            )
            if df is not None and not df.empty:
                return df
        except Exception as exc:
            if attempt < retries:
                time.sleep(delay * attempt)
            else:
                logger.warning(
                    "[%s] ошибка после %d попыток: %s",
                    ticker, retries, exc,
                )
    return pd.DataFrame()


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Разворачивает MultiIndex колонки в плоские строки."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(filter(None, map(str, c))).strip()
            for c in df.columns
        ]
    return df


def _get_close(df: pd.DataFrame, ticker: str):
    """Извлекает колонку Close из DataFrame yfinance."""
    df = _flatten_columns(df.copy())
    for col in [f"Close_{ticker}", "Close", ticker]:
        if col in df.columns:
            return df[col].rename(ticker)
    return None


def load_macro(start_date: str, end_date: str = None) -> pd.DataFrame:
    """Макро-индикаторы (1D, далее выравниваются через _align_daily)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Макро-индикаторы | %s → %s", start_date, end_date)

    data_dict = {}
    for ticker, name in MACRO_TICKERS.items():
        logger.info("  %-12s (%s)", ticker, name)
        raw = _safe_yf_download(ticker, start=start_date, end=end_date)
        series = _get_close(raw, ticker) if not raw.empty else None
        if series is not None and not series.empty:
            data_dict[ticker] = series
        time.sleep(0.3)

    if not data_dict:
        logger.error("Не удалось загрузить ни одного тикера.")
        return pd.DataFrame()

    df = pd.concat(list(data_dict.values()), axis=1)
    df = _normalize_index(df)
    df = df.dropna(how="all")

    df.index = df.index.round("1s")
    if not df.index.is_unique:
        df = df.groupby(level=0).last()

    result = pd.DataFrame(index=df.index)

    if "BTC-USD" in df:
        btc = df["BTC-USD"]
        result["btc_close"] = btc
        result["btc_pct_change"] = btc.pct_change() * 100
        result["btc_log_return"] = np.log(btc / btc.shift(1))

    if "SPY" in df:
        result["sp500_close"] = df["SPY"]
        result["sp500_pct"] = df["SPY"].pct_change() * 100

    if "DX-Y.NYB" in df:
        result["dxy_close"] = df["DX-Y.NYB"]
        result["dxy_pct"] = df["DX-Y.NYB"].pct_change() * 100

    if "^VIX" in df:
        result["vix_close"] = df["^VIX"]

    if "GLD" in df:
        result["gold_close"] = df["GLD"]
        if "btc_close" in result:
            result["btc_gold_ratio"] = (
                result["btc_close"] / result["gold_close"]
            )

    if "TLT" in df:
        result["tlt_close"] = df["TLT"]
        result["tlt_pct"] = df["TLT"].pct_change() * 100

    if "IEF" in df:
        result["ief_close"] = df["IEF"]

    if "USO" in df:
        result["oil_close"] = df["USO"]
        result["oil_pct"] = df["USO"].pct_change() * 100

    if "^TNX" in df:
        result["treasury_yield_10y"] = df["^TNX"]

    if "tlt_close" in result and "ief_close" in result:
        result["tlt_ief_spread"] = result["tlt_close"] - result["ief_close"]

    if "btc_pct_change" in result and "sp500_pct" in result:
        for w in (30, 90):
            result[f"btc_spy_corr_{w}d"] = (
                result["btc_pct_change"]
                .rolling(w, min_periods=max(w // 2, 1))
                .corr(result["sp500_pct"])
            )

    if "btc_pct_change" in result and "dxy_pct" in result:
        for w in (30, 90):
            result[f"btc_dxy_corr_{w}d"] = (
                result["btc_pct_change"]
                .rolling(w, min_periods=max(w // 2, 1))
                .corr(result["dxy_pct"])
            )

    if "btc_close" in result and "sp500_close" in result:
        result["btc_spy_ratio"] = (
            result["btc_close"] / result["sp500_close"]
        )

    if "btc_close" in result:
        for w in (7, 30, 90, 200):
            result[f"btc_sma_{w}d"] = (
                result["btc_close"].rolling(w, min_periods=1).mean()
            )
        result["btc_volatility_30d"] = (
            result["btc_pct_change"].rolling(30, min_periods=1).std()
        )

    if "^MOVE" in df:
        move = df["^MOVE"]
        result["move_index"] = move
        result["move_pct_change"] = move.pct_change() * 100
        move_mean = move.rolling(30, min_periods=10).mean()
        move_std = (
            move.rolling(30, min_periods=10).std().replace(0, np.nan)
        )
        result["move_z_score"] = (move - move_mean) / move_std
        if "vix_close" in result:
            vix = result["vix_close"]
            vix_mean = vix.rolling(30, min_periods=10).mean()
            vix_std = (
                vix.rolling(30, min_periods=10).std().replace(0, np.nan)
            )
            vix_z = (vix - vix_mean) / vix_std
            result["move_vix_spread"] = result["move_z_score"] - vix_z
    else:
        logger.warning("^MOVE не загружен. Признаки move_* пропущены.")

    if not result.index.is_unique:
        result = result.groupby(level=0).last()

    logger.info(
        "Макро готово: %d дней, %d колонок", len(result), len(result.columns),
    )
    return result


# Fear & Greed

def load_fear_greed(start_date: str) -> pd.DataFrame:
    """Crypto Fear & Greed Index (alternative.me)."""
    start_ms = _to_ms(start_date)
    logger.info("Fear & Greed Index | %s → сегодня", start_date)
    try:
        resp = _get_session().get(
            "https://api.alternative.me/fng/?limit=0", timeout=30,
        )
        resp.raise_for_status()
        raw = resp.json().get("data", [])
    except Exception as exc:
        logger.error("Fear & Greed API ошибка: %s", exc)
        return pd.DataFrame()

    if not raw:
        logger.warning("Fear & Greed: пустой ответ")
        return pd.DataFrame()

    df = pd.DataFrame(raw)
    df["timestamp_ms"] = df["timestamp"].astype(int) * 1000
    df = df[df["timestamp_ms"] >= start_ms]
    df = df[["timestamp_ms", "value"]].rename(
        columns={"value": "fear_greed_index"},
    )
    df["fear_greed_index"] = df["fear_greed_index"].astype(int)
    df = _ms_col_to_index(df, col="timestamp_ms")
    logger.info("Fear & Greed готово: %d дней", len(df))
    return df

# Open Interest — 1H

def load_open_interest(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """Open Interest с Bybit (только 1H)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Open Interest (1h) | %s → %s", start_date, end_date)

    endpoint = "https://api.bybit.com/v5/market/open-interest"
    records = _bybit_paginate(
        endpoint,
        {
            "category": "linear",
            "symbol": symbol,
            "intervalTime": "1h",
            "startTime": _to_ms(start_date),
            "endTime": _to_ms(end_date),
            "limit": 200,
        },
    )
    if not records:
        logger.warning("Open Interest: нет данных")
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df["timestamp"] = df["timestamp"].astype(int)
    df["open_interest"] = df["openInterest"].astype(float)
    df = df[["timestamp", "open_interest"]]
    df = _ms_col_to_index(df, col="timestamp")
    logger.info("Open Interest готово: %d записей", len(df))
    return df


# Funding Rate — 8H


def load_funding_rate(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """Funding Rate с Bybit (каждые 8 часов)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Funding Rate | %s → %s", start_date, end_date)

    start_ms = _to_ms(start_date)
    end_ms = _to_ms(end_date)
    all_records = []
    cur_start = start_ms

    while cur_start < end_ms:
        cur_end = min(cur_start + _FR_CHUNK_MS, end_ms)
        records = _bybit_paginate(
            _FR_ENDPOINT,
            {
                "category": "linear",
                "symbol": symbol,
                "startTime": cur_start,
                "endTime": cur_end,
                "limit": 200,
            },
        )
        if records:
            all_records.extend(records)
        cur_start = cur_end
        time.sleep(0.15)

    if not all_records:
        logger.warning("Funding Rate: нет данных")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    df["timestamp"] = df["fundingRateTimestamp"].astype(int)
    df["funding_rate"] = df["fundingRate"].astype(float)
    df = df[["timestamp", "funding_rate"]]
    df = _ms_col_to_index(df, col="timestamp")
    logger.info(
        "Funding Rate готово: %d записей | %s — %s",
        len(df), df.index[0].date(), df.index[-1].date(),
    )
    return df


# Long/Short Ratio — 1H


def load_long_short_ratio(
    start_date: str,
    end_date: str = None,
    symbol: str = "BTCUSDT",
) -> pd.DataFrame:
    """Long/Short Ratio трейдеров с Bybit (только 1H)."""
    if end_date is None:
        end_date = _end_date_default()
    logger.info("Long/Short Ratio (1h) | %s → %s", start_date, end_date)

    start_ms = _to_ms(start_date)
    end_ms = _to_ms(end_date)
    all_records = []
    cur_start = start_ms
    session = _get_session()

    while cur_start < end_ms:
        cur_end = min(cur_start + _LS_CHUNK_MS, end_ms)
        try:
            resp = session.get(
                "https://api.bybit.com/v5/market/account-ratio",
                params={
                    "category": "linear",
                    "symbol": symbol,
                    "period": "1h",
                    "limit": 500,
                    "startTime": cur_start,
                    "endTime": cur_end,
                },
                timeout=15,
            )
            resp.raise_for_status()
            data = resp.json()
            if data["retCode"] != 0:
                logger.error("L/S Ratio API: %s", data["retMsg"])
                break
            records = data["result"]["list"]
            if records:
                all_records.extend(records)
        except Exception as exc:
            logger.error("Ошибка L/S Ratio: %s", exc)
        cur_start = cur_end
        time.sleep(0.20)

    if not all_records:
        logger.warning("Long/Short Ratio: нет данных")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    df["timestamp"] = df["timestamp"].astype(int)
    df["buy_ratio"] = df["buyRatio"].astype(float)
    df["sell_ratio"] = df["sellRatio"].astype(float)
    df["ls_ratio"] = df["buy_ratio"] / df["sell_ratio"].replace(0, np.nan)
    df = df[["timestamp", "buy_ratio", "sell_ratio", "ls_ratio"]]
    df = _ms_col_to_index(df, col="timestamp")
    logger.info(
        "Long/Short Ratio готово: %d записей | %s — %s",
        len(df), df.index[0].date(), df.index[-1].date(),
    )
    return df


# Главная функция — load_all

def load_all(
    start_date: str = None,
    end_date: str = None,
    days: int = DOWNLOAD_DAYS,
) -> dict:
    if end_date is None:
        end_date = _end_date_default()
    if days is not None:
        start_date = (
            datetime.now(_MSK) - timedelta(days=days)
        ).strftime("%Y-%m-%d")

    _now_msk = datetime.now(_MSK)
    _end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    _start_dt = datetime.strptime(start_date, "%Y-%m-%d")

    if _end_dt.year > _now_msk.year + 1:
        raise RuntimeError(
            f"Подозрительная конечная дата: {end_date}. "
            f"Проверьте системные часы! Текущее MSK: "
            f"{_now_msk:%Y-%m-%d %H:%M:%S}"
        )
    if _start_dt > _end_dt:
        raise ValueError(
            f"start_date ({start_date}) позже end_date ({end_date})"
        )

    logger.info("load_all | %s → %s", start_date, end_date)

    result = {}

    result["candles_1h"] = load_candles_1h(start_date, end_date)
    if result["candles_1h"].empty:
        logger.error("candles_1h пуст — загрузка прервана")
        return result

    master_idx = result["candles_1h"].index

    result["candles_6h"] = load_candles_6h(start_date, end_date)

    result["bybit_daily"] = _align_daily(
        load_bybit_daily(start_date, end_date), master_idx,
    )
    result["macro"] = _align_daily(
        load_macro(start_date, end_date), master_idx,
    )
    result["fear_greed"] = _align_daily(
        load_fear_greed(start_date), master_idx,
    )

    result["open_interest"] = _align_intrabar(
        load_open_interest(start_date, end_date), master_idx,
    )
    result["long_short"] = _align_intrabar(
        load_long_short_ratio(start_date, end_date), master_idx,
    )

    result["funding_rate"] = _align_funding(
        load_funding_rate(start_date, end_date), master_idx,
    )

    logger.info("Итого загружено:")
    for k, v in result.items():
        if isinstance(v, pd.DataFrame) and not v.empty:
            logger.info(
                "  %-22s %7d строк | %s — %s",
                k, len(v), v.index[0].date(), v.index[-1].date(),
            )
        else:
            logger.info("  %-22s пусто", k)

    return result


# Сохранение

def save_all(data: dict, prefix: str = "") -> None:
    """Сохраняет все DataFrame из data в CSV-файлы."""
    stamp = datetime.now(_MSK).strftime("%Y%m%d_%H%M")
    p = f"{prefix}_" if prefix else ""
    for name, df in data.items():
        if isinstance(df, pd.DataFrame) and not df.empty:
            fname = f"{p}{name}_{stamp}.csv"
            df.to_csv(fname)
            logger.info("Сохранено: %s", fname)


if __name__ == "__main__":
    data = load_all()
    save_all(data, prefix="btc")

In [2]:
"""
feature_engineering.py — Построение признаков для классификации скачков BTC.

Принимает на вход dict из data_loader.load_all().
Мастер-индекс: 1H свечи (candles_1h).
Возвращает готовый DataFrame с признаками и таргетами.

Таргеты:
    target_1h : скачок цены через 1 бар (1H) на +/-threshold -> (-1, 0, 1)
    target_6h : скачок цены через 6 баров (6H) на +/-threshold -> (-1, 0, 1)
    Пороги по умолчанию: 0.8% (1H) и 1.5% (6H).
"""

import glob
import logging
import os
import re
import warnings
from datetime import datetime, timedelta
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# Константы


WARMUP_DAYS = 210
RECOMMENDED_DAYS_1H = 90
RECOMMENDED_DAYS_6H = 365

FORCE_DROP_6H: List[str] = [
    "ret_lag_1", "ret_lag_2", "ret_lag_3",
    "rsi_lag_1", "rsi_lag_2", "rsi_lag_3",
    "funding_lag_8h", "funding_lag_16h",
    "session_asia", "session_europe", "session_ny", "session_overlap",
    "dayofweek",
]

_PRICE_RE = re.compile(
    r"close|open|high|low|sma|ratio|vix|yield|spread|gold|oil|tlt|ief"
    r"|sp500|nasdaq|dxy|btc_spy|open_interest|fear_greed|turnover|volume"
    r"|move_index"
)
_PCT_RE = re.compile(
    r"pct|log_return|volatility|funding_rate|risk_regime|ls_ratio"
    r"|ls_buy|ls_sell|move_pct"
)
_CORR_RE = re.compile(r"corr")


# Технические индикаторы (вспомогательные функции)


def _rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Relative Strength Index."""
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=1).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=1).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)


def _ema(series: pd.Series, span: int) -> pd.Series:
    """Exponential Moving Average."""
    return series.ewm(span=span, adjust=False).mean()


def _atr(
    high: pd.Series,
    low: pd.Series,
    close: pd.Series,
    period: int = 14,
) -> pd.Series:
    """Average True Range."""
    prev_close = close.shift(1)
    tr = np.maximum(
        np.maximum(high - low, (high - prev_close).abs()),
        (low - prev_close).abs(),
    )
    return pd.Series(tr, index=close.index).rolling(
        period, min_periods=1,
    ).mean()


def _zscore(
    series: pd.Series,
    window: int,
    min_periods: int = 1,
) -> pd.Series:
    """Z-score нормализация с rolling-окном (std=0 → z=0)."""
    mean = series.rolling(window, min_periods=min_periods).mean()
    std = (
        series.rolling(window, min_periods=min_periods)
        .std()
        .replace(0, np.nan)
    )
    return ((series - mean) / std).fillna(0)


def _classify_movement(ret: pd.Series, threshold: float) -> pd.Series:
    """Классификация доходности: -1 / 0 / +1."""
    labels = pd.Series(0, index=ret.index, dtype=int)
    labels[ret > threshold] = 1
    labels[ret < -threshold] = -1
    return labels


# Объединение источников


def _merge_sources(data: dict) -> pd.DataFrame:
    """Объединяет все источники данных на мастер-индексе candles_1h."""
    candles = data.get("candles_1h", pd.DataFrame())
    if candles.empty:
        raise ValueError("candles_1h пуст — нет мастер-индекса")

    df = candles.copy()
    df.columns = [f"c_{c.lower()}" for c in df.columns]

    sources = {
        "d":  data.get("bybit_daily", pd.DataFrame()),
        "m":  data.get("macro", pd.DataFrame()),
        "fg": data.get("fear_greed", pd.DataFrame()),
        "oi": data.get("open_interest", pd.DataFrame()),
        "fr": data.get("funding_rate", pd.DataFrame()),
        "ls": data.get("long_short", pd.DataFrame()),
    }

    for prefix, src in sources.items():
        if src.empty:
            logger.warning("Источник '%s' пуст, пропускаем", prefix)
            continue
        src = src.copy()
        src.columns = [f"{prefix}_{c}" for c in src.columns]
        df = df.join(src, how="left")

    logger.info(
        "Объединено: %d строк, %d исходных колонок",
        len(df), len(df.columns),
    )
    return df


# Обработка пропусков


def _handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Заполняет пропуски по типу колонки."""
    for col in df.columns:
        col_l = col.lower()
        if _CORR_RE.search(col_l):
            df[col] = df[col].ffill().fillna(0)
        elif _PCT_RE.search(col_l):
            df[col] = df[col].fillna(0)
        elif _PRICE_RE.search(col_l):
            df[col] = df[col].ffill()

    df = df.dropna(subset=["c_close"])

    remaining_nan_cols = [c for c in df.columns if df[c].isna().any()]
    if remaining_nan_cols:
        for col in remaining_nan_cols:
            df[col] = df[col].ffill().fillna(0)
        logger.info(
            "Финальный fillna: обработано %d колонок с NaN: %s",
            len(remaining_nan_cols),
            remaining_nan_cols,
        )

    missing_pct = df.isnull().mean() * 100
    high_missing = missing_pct[missing_pct > 5]
    if not high_missing.empty:
        logger.warning(
            "Колонки с >5%% пропусков после fillna: %s",
            high_missing.round(1).to_dict(),
        )
    else:
        logger.info("Пропуски обработаны, критических колонок нет")

    logger.info("После обработки пропусков: %d строк", len(df))
    return df


# Технические признаки


def _add_technical(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет технические индикаторы на основе OHLCV."""
    close = df["c_close"]
    high = df["c_high"]
    low = df["c_low"]

    for p in [14, 24, 48]:
        df[f"rsi_{p}"] = _rsi(close, p)

    macd_line = _ema(close, 12) - _ema(close, 26)
    macd_signal = _ema(macd_line, 9)
    df["macd"] = macd_line
    df["macd_hist"] = macd_line - macd_signal

    sma24 = close.rolling(24, min_periods=1).mean()
    std24 = close.rolling(24, min_periods=1).std()
    bb_upper = sma24 + 2 * std24
    bb_lower = sma24 - 2 * std24
    bb_width = (bb_upper - bb_lower) / sma24.replace(0, np.nan)
    df["bb_pos"] = (
        (close - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)
    ).fillna(0.5)
    df["bb_width"] = bb_width.fillna(0)
    df["bb_squeeze"] = (
        bb_width < bb_width.rolling(168, min_periods=1).mean()
    ).astype(int)

    atr14 = _atr(high, low, close, 14)
    df["atr_pct"] = atr14 / close
    df["atr_ratio"] = (
        atr14 / atr14.rolling(168, min_periods=1).mean().replace(0, np.nan)
    ).fillna(1.0)

    bars = {"1d": 24, "4d": 96, "7d": 168, "28d": 672}
    for label, w in bars.items():
        df[f"sma_{label}"] = close.rolling(w, min_periods=1).mean()

    df["price_to_sma_1d"] = close / df["sma_1d"]
    df["price_to_sma_4d"] = close / df["sma_4d"]
    df["price_to_sma_7d"] = close / df["sma_7d"]
    df["price_to_sma_28d"] = close / df["sma_28d"]
    df["sma_7d_to_28d"] = df["sma_7d"] / df["sma_28d"]

    rsi24 = df["rsi_24"]
    rsi_min = rsi24.rolling(24, min_periods=1).min()
    rsi_max = rsi24.rolling(24, min_periods=1).max()
    df["stoch_rsi"] = (
        (rsi24 - rsi_min) / (rsi_max - rsi_min).replace(0, np.nan)
    ).fillna(0.5)

    log_ret = np.log(close / close.shift(1))
    df["volatility_1d"] = log_ret.rolling(24, min_periods=1).std()
    df["volatility_7d"] = log_ret.rolling(168, min_periods=1).std()
    df["volatility_28d"] = log_ret.rolling(672, min_periods=1).std()
    df["volatility_regime"] = (
        df["volatility_7d"] / df["volatility_28d"].replace(0, np.nan)
    ).fillna(1.0)

    df = df.drop(columns=["sma_1d", "sma_4d", "sma_7d", "sma_28d"])

    logger.info("Технические индикаторы добавлены")
    return df


# Объёмные признаки

def _add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет признаки объёма, VWAP и taker-volume."""
    vol = df["c_volume"]
    close = df["c_close"]

    vol_mean = vol.rolling(168, min_periods=1).mean()
    df["volume_ratio"] = (vol / vol_mean.replace(0, np.nan)).fillna(1.0)
    df["volume_spike"] = (df["volume_ratio"] > 2.0).astype(int)

    vwap = (
        (close * vol).rolling(24, min_periods=1).sum()
        / vol.rolling(24, min_periods=1).sum().replace(0, np.nan)
    )
    df["vwap_dist"] = ((close - vwap) / vwap.replace(0, np.nan)).fillna(0)

    price_dir = np.sign(close.pct_change())
    vol_dir = np.sign(vol.pct_change())
    df["vol_price_divergence"] = (price_dir != vol_dir).astype(int)

    if "d_turnover" in df.columns:
        d_turn_mean = df["d_turnover"].rolling(30, min_periods=1).mean()
        df["d_turnover_ratio"] = (
            df["d_turnover"] / d_turn_mean.replace(0, np.nan)
        ).fillna(1.0)
        df["d_turnover_ratio_lag1"] = df["d_turnover_ratio"].shift(1)
        df["d_turnover_ratio_ma3"] = (
            df["d_turnover_ratio"].rolling(3, min_periods=1).mean()
        )

    if "c_taker_buy_vol" in df.columns and "c_taker_sell_vol" in df.columns:
        buy = df["c_taker_buy_vol"]
        sell = df["c_taker_sell_vol"]
        df["taker_buy_zscore"] = _zscore(buy, 168)
        df["taker_sell_zscore"] = _zscore(sell, 168)
        total_taker = buy + sell
        df["taker_vol_imbalance"] = (
            (buy - sell) / total_taker.replace(0, np.nan)
        ).fillna(0)

    if "c_taker_vol_ratio" in df.columns:
        df["taker_vol_ratio_ma"] = (
            df["c_taker_vol_ratio"].rolling(24, min_periods=1).mean()
        )
        df["taker_vol_ratio_change"] = (
            df["c_taker_vol_ratio"].diff(6).fillna(0)
        )

    logger.info("Признаки объёма и taker volume добавлены")
    return df


# Свечные паттерны


def _add_candle_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет признаки свечных паттернов."""
    open_ = df["c_open"]
    high = df["c_high"]
    low = df["c_low"]
    close = df["c_close"]

    body = close - open_
    body_abs = body.abs()
    candle_range = (high - low).replace(0, np.nan)

    df["body_ratio"] = (body_abs / candle_range).fillna(0)
    df["body_dir"] = np.sign(body)

    oc_max = np.maximum(open_, close)
    oc_min = np.minimum(open_, close)
    df["upper_wick_ratio"] = ((high - oc_max) / candle_range).fillna(0)
    df["lower_wick_ratio"] = ((oc_min - low) / candle_range).fillna(0)

    df["doji"] = (df["body_ratio"] < 0.1).astype(int)
    df["hl_pct"] = ((high - low) / close).fillna(0)

    logger.info("Свечные паттерны добавлены")
    return df


# Макро-признаки


def _add_macro_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет макро-признаки.

    ВАЖНО: все m_* колонки УЖЕ сдвинуты на 24 часа в data_loader._align_daily,
    поэтому никаких дополнительных shift(24) здесь не делаем.
    """
    close = df["c_close"]

    if "m_btc_sma_90d" in df.columns:
        df["btc_to_macro_sma90"] = (
            close / df["m_btc_sma_90d"].replace(0, np.nan)
        ).fillna(1.0)

    ema200_1h = close.ewm(span=4800, min_periods=1).mean()
    df["btc_to_ema200_1h"] = close / ema200_1h

    if "m_vix_close" in df.columns:
        df["vix_high"] = (df["m_vix_close"] > 30).astype(int)

    if "fg_fear_greed_index" in df.columns:
        fg = df["fg_fear_greed_index"]
        df["fg_extreme_fear"] = (fg < 25).astype(int)
        df["fg_fear"] = ((fg >= 25) & (fg < 45)).astype(int)
        df["fg_greed"] = ((fg >= 55) & (fg < 75)).astype(int)

    if "m_tlt_ief_spread" in df.columns:
        df["yield_spread_change"] = (
            df["m_tlt_ief_spread"].diff(24).fillna(0)
        )

    if "m_treasury_yield_10y" in df.columns:
        y = df["m_treasury_yield_10y"]
        df["yield_vs_mean"] = (
            y - y.rolling(1440, min_periods=24).mean()
        ).fillna(0)

    logger.info("Макро признаки добавлены (lag учтён в data_loader)")
    return df


# Фьючерсные признаки


def _add_futures_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет признаки Open Interest, Funding Rate, Long/Short Ratio."""
    if "oi_open_interest" in df.columns:
        oi = df["oi_open_interest"]
        df["oi_change_pct"] = oi.pct_change().fillna(0) * 100
        df["oi_change_6bar"] = oi.pct_change(6).fillna(0) * 100
        oi_mean = oi.rolling(168, min_periods=1).mean()
        df["oi_ratio"] = (oi / oi_mean.replace(0, np.nan)).fillna(1.0)

        price_dir = np.sign(df["c_close"].pct_change())
        oi_dir = np.sign(df["oi_change_pct"])
        df["oi_price_divergence"] = (price_dir != oi_dir).astype(int)

    if "fr_funding_rate" in df.columns:
        fr = df["fr_funding_rate"].fillna(0)
        df["funding_rate_abs"] = fr.abs()
        df["funding_change"] = fr.diff(8).fillna(0)

    if "ls_buy_ratio" in df.columns:
        br = df["ls_buy_ratio"].fillna(0.5)
        df["ls_buy_ratio"] = br
        df["ls_long_dominant"] = (br > 0.65).astype(int)
        if "ls_ls_ratio" in df.columns:
            ls = df["ls_ls_ratio"].fillna(1)
            df["ls_ratio_change"] = ls.diff().fillna(0)
            df["ls_ratio_vs_mean"] = (
                ls / ls.rolling(42, min_periods=1).mean().replace(0, np.nan)
            ).fillna(1.0)
        else:
            logger.warning("ls_ls_ratio не найден, пропускаем ratio-признаки")
    else:
        logger.warning(
            "Long/Short Ratio: данные отсутствуют (ls_buy_ratio не найден)"
        )

    logger.info("Фьючерсные признаки добавлены (OI + FR + L/S)")
    return df



# Лаговые признаки


def _add_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет лаговые признаки.

    Для funding используем лаги 8h и 16h — они попадают в предыдущие
    funding-периоды, а не дублируют текущее ffill-значение.
    """
    log_ret = np.log(df["c_close"] / df["c_close"].shift(1)).fillna(0)

    for lag in [1, 2, 3, 6, 12, 24]:
        df[f"ret_lag_{lag}"] = log_ret.shift(lag).fillna(0)

    if "fr_funding_rate" in df.columns:
        for lag in [8, 16]:
            df[f"funding_lag_{lag}h"] = (
                df["fr_funding_rate"].fillna(0).shift(lag).fillna(0)
            )

    for lag in [1, 2, 3]:
        if "rsi_24" in df.columns:
            df[f"rsi_lag_{lag}"] = df["rsi_24"].shift(lag).fillna(50.0)

    df["cum_ret_1d"] = (
        log_ret.rolling(24, min_periods=1).sum().shift(1).fillna(0)
    )
    df["cum_ret_7d"] = (
        log_ret.rolling(168, min_periods=1).sum().shift(1).fillna(0)
    )

    logger.info("Лаговые признаки добавлены")
    return df


# Временные признаки


def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет временные признаки (час, день недели, сессии)."""
    idx = df.index

    df["hour"] = idx.hour
    df["dayofweek"] = idx.dayofweek
    df["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    df["month"] = idx.month

    df["session_asia"] = ((idx.hour >= 0) & (idx.hour < 8)).astype(int)
    df["session_europe"] = ((idx.hour >= 7) & (idx.hour < 16)).astype(int)
    df["session_ny"] = ((idx.hour >= 13) & (idx.hour < 22)).astype(int)
    df["session_overlap"] = ((idx.hour >= 13) & (idx.hour < 16)).astype(int)

    logger.info("Временные признаки добавлены")
    return df


# Производные моментум-признаки


def _add_derived_momentum(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет производные моментум-признаки."""
    if "rsi_24" in df.columns:
        df["rsi_velocity"] = df["rsi_24"].diff(6).fillna(0)
        df["rsi_acceleration"] = df["rsi_velocity"].diff(6).fillna(0)

    if "fr_funding_rate" in df.columns and "oi_change_pct" in df.columns:
        fr = df["fr_funding_rate"].fillna(0)
        oi = df["oi_change_pct"].fillna(0)
        df["funding_oi_combo"] = fr * oi

    if "volatility_1d" in df.columns and "c_close" in df.columns:
        log_ret = np.log(df["c_close"] / df["c_close"].shift(1))
        vol_28d_pure = (
            log_ret.shift(24).rolling(672, min_periods=48).std()
        )
        vol_norm = vol_28d_pure.replace(0, np.nan)
        df["vol_z_score"] = (
            (df["volatility_1d"] - vol_28d_pure) / vol_norm
        ).fillna(0)

    if "price_to_sma_7d" in df.columns and "volatility_7d" in df.columns:
        vol7 = df["volatility_7d"].replace(0, np.nan)
        df["trend_strength"] = (
            (df["price_to_sma_7d"] - 1).abs() / vol7
        ).fillna(0)

    if "fg_fear_greed_index" in df.columns:
        df["fg_reversal_signal"] = (
            df["fg_fear_greed_index"].diff(24).fillna(0)
        )

    if "volume_ratio" in df.columns and "ret_lag_1" in df.columns:
        df["vol_price_confirm"] = (
            df["volume_ratio"] * df["ret_lag_1"].abs()
        ).fillna(0)

    if "ls_buy_ratio" in df.columns:
        df["ls_acceleration"] = df["ls_buy_ratio"].diff(6).fillna(0)

    if "ret_lag_1" in df.columns:
        log_ret = df["ret_lag_1"]
        ret_std = log_ret.shift(1).rolling(672, min_periods=24).std()
        big_move = (log_ret.abs() > 2 * ret_std).astype(int).fillna(0)
        grp = big_move.groupby(big_move.cumsum())
        df["bars_since_big_move"] = (
            grp.cumcount().where(big_move == 0, 0).clip(upper=672)
        )

    logger.info("Производные моментум-признаки добавлены")
    return df


# Финальный проход заполнения NaN

def _final_fillna(df: pd.DataFrame) -> pd.DataFrame:
    """
    Финальный проход заполнения NaN перед таргетами.

    Только ffill → fillna(0). bfill УБРАН — он подмешивал будущие
    значения в начало ряда. Warmup-период должен быть отрезан в
    pipeline'ах build_features_1h / _6h.
    """
    skip_cols = {
        "c_close", "target_1h", "target_6h",
        "fwd_ret_1h", "fwd_ret_6h",
    }
    nan_before = df.isnull().sum().sum()

    for col in df.columns:
        if col in skip_cols:
            continue
        if df[col].isna().any():
            df[col] = df[col].ffill().fillna(0)

    nan_after = df.isnull().sum().sum()
    if nan_before > 0:
        logger.info(
            "_final_fillna: устранено %d NaN (осталось %d)",
            nan_before - nan_after, nan_after,
        )
    return df


# Таргеты


def _add_targets(
    df: pd.DataFrame,
    threshold_1h: float = 0.008,
    threshold_6h: float = 0.015,
) -> pd.DataFrame:
    """Добавляет целевые переменные для 1H и 6H горизонтов."""
    close = df["c_close"]

    future_ret_1h = close.shift(-1) / close - 1
    future_ret_6h = close.shift(-6) / close - 1

    df["fwd_ret_1h"] = future_ret_1h
    df["fwd_ret_6h"] = future_ret_6h
    df["target_1h"] = _classify_movement(future_ret_1h, threshold_1h)
    df["target_6h"] = _classify_movement(future_ret_6h, threshold_6h)

    df = df.dropna(
        subset=["target_1h", "target_6h", "fwd_ret_1h", "fwd_ret_6h"]
    )

    logger.info(
        "Таргеты 1H (+-%.1f%%): %s",
        threshold_1h * 100,
        df["target_1h"].value_counts().sort_index().to_dict(),
    )
    logger.info(
        "Таргеты 6H (+-%.1f%%): %s",
        threshold_6h * 100,
        df["target_6h"].value_counts().sort_index().to_dict(),
    )
    return df



# Удаление leakage-колонок


# Ценовой контекст (стационарные производные от c_close)


def _add_price_context(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет стационарные ценовые производные.

        price_log         — log(close), медленный тренд режима
        price_zscore_90d  — (close - sma_90d) / std_90d
        price_rank_90d    — pct-rank close в окне ~90 дней, [0, 1]
        price_vs_sma_200d — close / sma_200d - 1

    Все 4 колонки стационарны и НЕ являются утечкой данных: основаны на c_close в
    момент t, без обращения к будущему.
    """
    if "c_close" not in df.columns:
        logger.warning("_add_price_context: нет c_close — шаг пропущен")
        return df

    close = df["c_close"]

    win_90  = min(2160, max(60, len(df) // 8))
    win_200 = min(4800, max(120, len(df) // 4))

    df["price_log"] = np.log(close.clip(lower=1e-9))

    sma_90 = close.rolling(win_90, min_periods=win_90 // 4).mean()
    std_90 = close.rolling(win_90, min_periods=win_90 // 4).std()
    df["price_zscore_90d"] = (
        ((close - sma_90) / std_90)
        .replace([np.inf, -np.inf], np.nan)
    )

    df["price_rank_90d"] = (
        close.rolling(win_90, min_periods=win_90 // 4).rank(pct=True)
    )

    sma_200 = close.rolling(win_200, min_periods=win_200 // 4).mean()
    df["price_vs_sma_200d"] = (
        (close / sma_200 - 1.0)
        .replace([np.inf, -np.inf], np.nan)
    )

    n_added = sum(
        1 for c in ["price_log", "price_zscore_90d",
                    "price_rank_90d", "price_vs_sma_200d"]
        if c in df.columns
    )
    logger.info("_add_price_context: добавлено %d price_* колонок", n_added)
    return df


# Удаление колонок с утечкой данных


def _drop_leakage(df: pd.DataFrame) -> pd.DataFrame:
    """Удаляет абсолютные цены, дубликаты и нулевые-сигнальные колонки."""
    absolute_price_cols = [
        "c_open", "c_high", "c_low",
        "c_volume", "c_turnover",
        "c_taker_buy_vol", "c_taker_sell_vol",
        "m_btc_close", "m_sp500_close",
        "m_dxy_close", "m_gold_close", "m_tlt_close",
        "m_ief_close", "m_oil_close",
        "m_btc_sma_7d", "m_btc_sma_30d",
        "m_btc_sma_90d", "m_btc_sma_200d",
        "m_btc_spy_ratio",
        "d_open", "d_high", "d_low", "d_close",
        "d_volume",
        "oi_open_interest",
        "ls_sell_ratio", "ls_ls_ratio",
    ]
    confirmed_duplicates = [
        "m_btc_log_return",
        "macd_signal",
    ]
    legacy_cols = [
        "move_z_score",
        "move_vix_spread",
        "ls_short_dominant",
        "candle_strength",
        "vwap_dist_tmp",
    ]
    zero_signal_cols = [
        "m_btc_spy_corr_30d", "m_btc_spy_corr_90d",
        "m_btc_dxy_corr_30d", "m_btc_dxy_corr_90d",
        "m_btc_volatility_30d",
        "d_turnover",
        "vix_medium",
    ]

    all_to_drop = (
        absolute_price_cols
        + confirmed_duplicates
        + legacy_cols
        + zero_signal_cols
    )
    to_drop = [c for c in all_to_drop if c in df.columns]
    df = df.drop(columns=to_drop)

    n_price = len([c for c in absolute_price_cols if c in to_drop])
    n_dupes = len([
        c for c in confirmed_duplicates + legacy_cols if c in to_drop
    ])
    n_zero = len([c for c in zero_signal_cols if c in to_drop])
    logger.info(
        "Удалено %d колонок: %d абсолютных цен, %d дубликатов, "
        "%d нулевого сигнала",
        len(to_drop), n_price, n_dupes, n_zero,
    )
    return df


# Warmup cut


def _trim_warmup(df: pd.DataFrame, warmup_bars: int) -> pd.DataFrame:
    """Отбрасывает первые warmup_bars (rolling-окна ещё накапливаются)."""
    if len(df) > warmup_bars + 100:
        df = df.iloc[warmup_bars:].copy()
        logger.info("Отброшен warmup: %d баров", warmup_bars)
    else:
        logger.warning(
            "Warmup cut пропущен: данных слишком мало (%d <= %d)",
            len(df), warmup_bars + 100,
        )
    return df


# Главная функция построения признаков


def build_features(
    data: dict,
    threshold_1h: float = 0.008,
    threshold_6h: float = 0.015,
) -> Tuple[pd.DataFrame, List[str]]:
    """Строит финальный датасет признаков из словаря источников данных."""
    df = _merge_sources(data)
    df = _handle_missing(df)
    df = _add_technical(df)
    df = _add_volume_features(df)
    df = _add_candle_features(df)
    df = _add_macro_features(df)
    df = _add_futures_features(df)
    df = _add_lags(df)
    df = _add_time_features(df)
    df = _add_derived_momentum(df)
    df = _final_fillna(df)
    df = _add_targets(df, threshold_1h=threshold_1h, threshold_6h=threshold_6h)
    df = _add_price_context(df),
    df = _drop_leakage(df)

    feature_cols = [
        c for c in df.columns
        if c not in ("target_1h", "target_6h", "fwd_ret_1h", "fwd_ret_6h")
    ]
    logger.info(
        "build_features: %d строк, %d признаков", len(df), len(feature_cols),
    )
    return df, feature_cols


# CSV-утилиты


def check_csv_files(
    csv_dir: str = ".",
    prefix: str = "btc",
) -> dict:
    """Ищет сохранённые CSV-файлы по ключам источников."""
    source_keys = [
        "candles_1h", "bybit_daily", "macro", "fear_greed",
        "open_interest", "funding_rate", "long_short",
    ]
    result = {}
    logger.info("Поиск CSV-файлов в '%s' (prefix='%s')", csv_dir, prefix)
    for key in source_keys:
        pattern = os.path.join(csv_dir, f"{prefix}_{key}_*.csv")
        matches = sorted(glob.glob(pattern))
        if matches:
            result[key] = matches[-1]
            logger.info(
                "  OK %-20s -> %s", key, os.path.basename(matches[-1]),
            )
        else:
            result[key] = None
            logger.warning(
                "  MISS %-20s -> не найден (%s)", key, pattern,
            )

    found = [k for k, v in result.items() if v is not None]
    missing = [k for k, v in result.items() if v is None]
    logger.info(
        "Итог: найдено %d/%d файлов%s",
        len(found), len(source_keys),
        (f", отсутствуют: {missing}" if missing else ""),
    )
    return result


def load_from_csv(csv_dir: str = ".", prefix: str = "btc") -> dict:
    """Загружает все источники из CSV-файлов."""
    csv_map = check_csv_files(csv_dir, prefix)
    data = {}

    for key, filepath in csv_map.items():
        if filepath is None:
            data[key] = pd.DataFrame()
            continue
        try:
            df = pd.read_csv(filepath, index_col=0, parse_dates=True)
            if hasattr(df.index, "tz") and df.index.tz is not None:
                df.index = df.index.tz_convert(None)
            data[key] = df
            logger.info("Загружен %-20s: %d строк", key, len(df))
        except Exception as exc:
            logger.error("Ошибка загрузки %s: %s", filepath, exc)
            data[key] = pd.DataFrame()

    return data


def _api_load(start_date: Optional[str] = None) -> dict:
    """Загружает данные через API (data_loader.load_all)."""
    from data_loader import load_all  # noqa: PLC0415

    if start_date is None:
        start_date = (
            datetime.now() - timedelta(days=WARMUP_DAYS + RECOMMENDED_DAYS_6H)
        ).strftime("%Y-%m-%d")
    logger.info("API-загрузка данных с %s ...", start_date)
    return load_all(start_date=start_date)


def smart_load_data(
    csv_dir: str = ".",
    prefix: str = "btc",
    start_date: Optional[str] = None,
    force_api: bool = False,
) -> dict:
    """Умная загрузка: CSV если есть, иначе API."""
    if force_api:
        logger.info("smart_load_data: force_api=True -> API")
        return _api_load(start_date)

    csv_map = check_csv_files(csv_dir, prefix)
    candles_ok = csv_map.get("candles_1h") is not None
    missing = [k for k, v in csv_map.items() if v is None]

    if candles_ok:
        if missing:
            logger.warning(
                "smart_load_data: candles_1h найден, отсутствуют: %s. "
                "Загружаем из CSV.",
                missing,
            )
        else:
            logger.info("smart_load_data: все CSV найдены -> load_from_csv()")
        return load_from_csv(csv_dir, prefix)

    logger.warning("smart_load_data: candles_1h не найден -> API-загрузка")
    if start_date is None:
        start_date = (
            datetime.now() - timedelta(days=WARMUP_DAYS + RECOMMENDED_DAYS_6H)
        ).strftime("%Y-%m-%d")
        logger.info("start_date не задан, дефолт: %s", start_date)
    return _api_load(start_date)


# ---------------------------------------------------------------------------
# Pipeline для 1H
# ---------------------------------------------------------------------------

def build_features_1h(
    data: Optional[dict] = None,
    days: int = RECOMMENDED_DAYS_1H,
    threshold: float = 0.008,
    csv_dir: str = ".",
    prefix: str = "btc",
    force_api: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """Полный pipeline для 1H датасета (с warmup cut)."""
    if data is None:
        total_days = days + WARMUP_DAYS
        start_date = (
            datetime.now() - timedelta(days=total_days)
        ).strftime("%Y-%m-%d")
        logger.info(
            "1H pipeline: данные с %s (%d + %d = %d дней)",
            start_date, days, WARMUP_DAYS, total_days,
        )
        data = smart_load_data(
            csv_dir=csv_dir, prefix=prefix,
            start_date=start_date, force_api=force_api,
        )

    df, feature_cols = build_features(
        data, threshold_1h=threshold, threshold_6h=0.015,
    )
    if df.empty:
        logger.error("1H: build_features вернул пустой DataFrame")
        return df, []

    # Warmup cut на 1H: WARMUP_DAYS * 24 баров.
    df = _trim_warmup(df, warmup_bars=WARMUP_DAYS * 24)

    if df.empty:
        logger.error("1H: после warmup cut DataFrame пуст")
        return df, []

    actual_days = (df.index.max() - df.index.min()).days
    if actual_days < days * 0.9:
        logger.warning(
            "1H: запрошено %d дней, получено %d.", days, actual_days,
        )

    target_and_ret = ["target_1h", "fwd_ret_1h"]
    keep = [c for c in feature_cols if c in df.columns] + [
        c for c in target_and_ret if c in df.columns
    ]
    df = df[keep].copy()
    df = df.rename(columns={"target_1h": "target", "fwd_ret_1h": "fwd_ret"})
    feature_cols_out = [
        c for c in df.columns if c not in ("target", "fwd_ret")
    ]

    logger.info(
        "1H датасет: %d строк, %d признаков | %s - %s (%d дней)",
        len(df), len(feature_cols_out),
        df.index[0].date(), df.index[-1].date(), actual_days,
    )
    logger.info(
        "  target balance: %s",
        df["target"].value_counts().sort_index().to_dict(),
    )
    return df, feature_cols_out


# ---------------------------------------------------------------------------
# Pipeline для 6H
# ---------------------------------------------------------------------------

def build_features_6h(
    data: Optional[dict] = None,
    days: int = RECOMMENDED_DAYS_6H,
    threshold: float = 0.015,
    csv_dir: str = ".",
    prefix: str = "btc",
    force_api: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """Полный pipeline для 6H датасета (с warmup cut)."""
    if data is None:
        total_days = days + WARMUP_DAYS
        start_date = (
            datetime.now() - timedelta(days=total_days)
        ).strftime("%Y-%m-%d")
        logger.info(
            "6H pipeline: данные с %s (%d + %d = %d дней)",
            start_date, days, WARMUP_DAYS, total_days,
        )
        data = smart_load_data(
            csv_dir=csv_dir, prefix=prefix,
            start_date=start_date, force_api=force_api,
        )

    df, feature_cols = build_features(
        data, threshold_1h=0.008, threshold_6h=threshold,
    )
    if df.empty:
        logger.error("6H: build_features вернул пустой DataFrame")
        return df, []

    # Warmup cut на 1H-индексе ДО downsampling: WARMUP_DAYS * 24.
    df = _trim_warmup(df, warmup_bars=WARMUP_DAYS * 24)

    if df.empty:
        logger.error("6H: после warmup cut DataFrame пуст")
        return df, []

    actual_days = (df.index.max() - df.index.min()).days
    if actual_days < days * 0.9:
        logger.warning(
            "6H: запрошено %d дней, получено %d.", days, actual_days,
        )

    target_and_ret = ["target_6h", "fwd_ret_6h"]
    keep = [c for c in feature_cols if c in df.columns] + [
        c for c in target_and_ret if c in df.columns
    ]
    df_6h = df[keep].iloc[::6].copy()

    idx_6h = df_6h.index
    df_6h["hour"] = idx_6h.hour
    df_6h["is_weekend"] = (idx_6h.dayofweek >= 5).astype(int)
    df_6h["month"] = idx_6h.month

    df_6h = df_6h.rename(
        columns={"target_6h": "target", "fwd_ret_6h": "fwd_ret"},
    )
    df_6h = df_6h.dropna(subset=["target"])

    if "oi_change_6bar" in df_6h.columns:
        if "oi_change_pct" in df_6h.columns:
            df_6h = df_6h.drop(columns=["oi_change_pct"])
        df_6h = df_6h.rename(columns={"oi_change_6bar": "oi_change_pct"})

    if "bars_since_big_move" in df_6h.columns:
        df_6h["bars_since_big_move"] = (
            (df_6h["bars_since_big_move"] / 6)
            .round()
            .astype(int)
            .clip(upper=112)
        )

    to_drop_6h = [c for c in FORCE_DROP_6H if c in df_6h.columns]
    if to_drop_6h:
        df_6h = df_6h.drop(columns=to_drop_6h)
        logger.info(
            "6H: удалены sub-hour/константные колонки: %s", to_drop_6h,
        )

    rename_map = {}
    if "ret_lag_6" in df_6h.columns:
        rename_map["ret_lag_6"] = "ret_lag_1_6h"
    if "ret_lag_12" in df_6h.columns:
        rename_map["ret_lag_12"] = "ret_lag_2_6h"
    if "ret_lag_24" in df_6h.columns:
        rename_map["ret_lag_24"] = "ret_lag_4_6h"
    if rename_map:
        df_6h = df_6h.rename(columns=rename_map)

    feature_cols_6h = [
        c for c in df_6h.columns if c not in ("target", "fwd_ret")
    ]
    logger.info(
        "6H датасет: %d строк, %d признаков | %s - %s (%d дней)",
        len(df_6h), len(feature_cols_6h),
        df_6h.index[0].date(), df_6h.index[-1].date(), actual_days,
    )
    logger.info(
        "  target balance: %s",
        df_6h["target"].value_counts().sort_index().to_dict(),
    )
    return df_6h, feature_cols_6h



def build_features_jump_1h(
    data: Optional[dict] = None,
    days: int = RECOMMENDED_DAYS_1H,
    threshold: float = 0.008,
    csv_dir: str = ".",
    prefix: str = "btc",
    force_api: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """1H jump-датасет: только скачки + ценовой контекст."""
    df_full, feat_full = build_features_1h(
        data=data, days=days, threshold=threshold,
        csv_dir=csv_dir, prefix=prefix, force_api=force_api,
    )
    if df_full.empty:
        return df_full, []

    before = len(df_full)
    df_jump = df_full[df_full["target"] != 0].copy()
    after = len(df_jump)
    pct = 100.0 * after / before if before else 0.0
    logger.info(
        "1H JUMP: отфильтровано %d → %d баров (%.1f%% скачков)",
        before, after, pct,
    )

    n_price = sum(1 for c in feat_full if c.startswith("price_"))
    logger.info(
        "1H JUMP датасет: %d строк, %d признаков (price_*: %d)",
        len(df_jump), len(feat_full), n_price,
    )
    logger.info(
        "  jump balance: %s",
        df_jump["target"].value_counts().sort_index().to_dict(),
    )
    return df_jump, feat_full


def build_features_jump_6h(
    data: Optional[dict] = None,
    days: int = RECOMMENDED_DAYS_6H,
    threshold: float = 0.015,
    csv_dir: str = ".",
    prefix: str = "btc",
    force_api: bool = False,
) -> Tuple[pd.DataFrame, List[str]]:
    """6H jump-датасет: только скачки + ценовой контекст."""
    df_full, feat_full = build_features_6h(
        data=data, days=days, threshold=threshold,
        csv_dir=csv_dir, prefix=prefix, force_api=force_api,
    )
    if df_full.empty:
        return df_full, []

    before = len(df_full)
    df_jump = df_full[df_full["target"] != 0].copy()
    after = len(df_jump)
    pct = 100.0 * after / before if before else 0.0
    logger.info(
        "6H JUMP: отфильтровано %d → %d баров (%.1f%% скачков)",
        before, after, pct,
    )

    n_price = sum(1 for c in feat_full if c.startswith("price_"))
    logger.info(
        "6H JUMP датасет: %d строк, %d признаков (price_*: %d)",
        len(df_jump), len(feat_full), n_price,
    )
    logger.info(
        "  jump balance: %s",
        df_jump["target"].value_counts().sort_index().to_dict(),
    )
    return df_jump, feat_full


# Точка входа

if __name__ == "__main__":
    import gc
    import time as _time
    from datetime import timezone

    DAYS_1H = 720
    DAYS_6H = 720
    CSV_DIR = "."
    PREFIX = "btc"
    FORCE_API = True

    _MSK = timezone(timedelta(hours=3))

    total_days = max(DAYS_1H, DAYS_6H) + WARMUP_DAYS
    start_date = (
        datetime.now(_MSK) - timedelta(days=total_days)
    ).strftime("%Y-%m-%d")

    logger.info(
        "Единая загрузка: %d дней (max(%d, %d) + %d warmup)",
        total_days, DAYS_1H, DAYS_6H, WARMUP_DAYS,
    )

    t0 = _time.time()
    data = load_from_csv(csv_dir=CSV_DIR, prefix=PREFIX)
    t_load = _time.time() - t0
    logger.info("Загрузка завершена за %.0f сек", t_load)

    t0 = _time.time()
    df_1h, feat_1h = build_features_1h(
        data=data, days=DAYS_1H, threshold=0.008,
    )
    t_1h = _time.time() - t0

    t0 = _time.time()
    df_6h, feat_6h = build_features_6h(
        data=data, days=DAYS_6H, threshold=0.015,
    )
    t_6h = _time.time() - t0

    del data
    gc.collect()

    df_1h_jump = (
        df_1h[df_1h["target"] != 0].copy() if not df_1h.empty else df_1h
    )
    df_6h_jump = (
        df_6h[df_6h["target"] != 0].copy() if not df_6h.empty else df_6h
    )

    print("\n" + "=" * 62)
    print("СВОДКА")
    print("=" * 62)
    print(f"Загрузка: {t_load:.0f} сек")

    if not df_1h.empty:
        d1 = (df_1h.index[-1] - df_1h.index[0]).days
        fname_1h = "features_1h.csv"
        df_1h.to_csv(fname_1h)
        logger.info(
            "Сохранено %s | %s — %s",
            fname_1h, df_1h.index[0].date(), df_1h.index[-1].date(),
        )
        print(
            f"1H датасет:  {len(df_1h):>6} строк | "
            f"{len(feat_1h):>3} признаков | {d1} дней | {t_1h:.0f} сек"
        )
        print(
            f"  Период: {df_1h.index[0].date()} — {df_1h.index[-1].date()}"
        )
        print(
            f"  Таргет: "
            f"{df_1h['target'].value_counts().sort_index().to_dict()}"
        )
        print(f"  Файл  : {fname_1h}")

        # JUMP 1H
        if not df_1h_jump.empty:
            fname_1h_jump = "features_1h_jump.csv"
            df_1h_jump.to_csv(fname_1h_jump)
            n_price = sum(1 for c in feat_1h if c.startswith("price_"))
            pct = 100.0 * len(df_1h_jump) / len(df_1h)
            print(
                f"1H JUMP:     {len(df_1h_jump):>6} строк ({pct:.1f}% от 1H) | "
                f"price_*={n_price}"
            )
            print(
                f"  Jump:   "
                f"{df_1h_jump['target'].value_counts().sort_index().to_dict()}"
            )
            print(f"  Файл  : {fname_1h_jump}")
    else:
        print("1H датасет:  ПУСТО")

    print()

    if not df_6h.empty:
        d6 = (df_6h.index[-1] - df_6h.index[0]).days
        fname_6h = "features_6h.csv"
        df_6h.to_csv(fname_6h)
        logger.info(
            "Сохранено %s | %s — %s",
            fname_6h, df_6h.index[0].date(), df_6h.index[-1].date(),
        )
        print(
            f"6H датасет:  {len(df_6h):>6} строк | "
            f"{len(feat_6h):>3} признаков | {d6} дней | {t_6h:.0f} сек"
        )
        print(
            f"  Период: {df_6h.index[0].date()} — {df_6h.index[-1].date()}"
        )
        print(
            f"  Таргет: "
            f"{df_6h['target'].value_counts().sort_index().to_dict()}"
        )
        print(f"  Файл  : {fname_6h}")

        # JUMP 6H
        if not df_6h_jump.empty:
            fname_6h_jump = "features_6h_jump.csv"
            df_6h_jump.to_csv(fname_6h_jump)
            n_price = sum(1 for c in feat_6h if c.startswith("price_"))
            pct = 100.0 * len(df_6h_jump) / len(df_6h)
            print(
                f"6H JUMP:     {len(df_6h_jump):>6} строк ({pct:.1f}% от 6H) | "
                f"price_*={n_price}"
            )
            print(
                f"  Jump:   "
                f"{df_6h_jump['target'].value_counts().sort_index().to_dict()}"
            )
            print(f"  Файл  : {fname_6h_jump}")
    else:
        print("6H датасет:  ПУСТО")

    if not df_1h.empty and not df_6h.empty:
        print(
            f"\nСоотношение строк 1H/6H ~ "
            f"{len(df_1h) / len(df_6h):.1f}:1  (теория 6:1)"
        )

    print("=" * 62)

    del df_1h, df_6h, feat_1h, feat_6h
    del df_1h_jump, df_6h_jump
    gc.collect()

ValueError: candles_1h пуст — нет мастер-индекса

In [3]:
"""
БЛОК 3 — Анализ признаков и формирование финального датасета

    - Для jump-датасетов: TARGET_COL = "fwd_ret" (регрессия, непрерывный),
      а не "target" (классификация). Скоринг: |ρ_Spearman(fwd_ret)|
      и MI рассчитываются относительно fwd_ret.

    - Защищённые колонки для jump-датасетов включают ценовой контекст:
        price_log, price_rank, price_vs_sma_ratio, price_zscore.

    - Точка входа запускает process() 4 раза, выводит сравнение 4-х ТФ.
=============================================================================
"""

import json
import os
from collections import defaultdict
from typing import Any, Literal

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression


# КОНФИГ


INPUT_1H      = "features_1h.csv"
INPUT_6H      = "features_6h.csv"
INPUT_1H_JUMP = "features_1h_jump.csv"
INPUT_6H_JUMP = "features_6h_jump.csv"

OUTPUT_1H      = "final_dataset_1h.csv"
OUTPUT_6H      = "final_dataset_6h.csv"
OUTPUT_1H_JUMP = "final_dataset_1h_jump.csv"
OUTPUT_6H_JUMP = "final_dataset_6h_jump.csv"


TARGET_CLF = "target"

TARGET_REG = "fwd_ret"

THRESHOLD_WEAK_RHO = 0.01
THRESHOLD_WEAK_MI  = 0.003

CORR_DUP_THRESHOLD = 0.65

SCORE_WEIGHT_RHO = 0.5
SCORE_WEIGHT_MI  = 0.5

MI_RANDOM_STATE = 42


_NON_FEATURE_CLF = {"target", "fwd_ret"}

_NON_FEATURE_REG = {"fwd_ret"}


KEEP_ALWAYS_CLF = [
    "c_close",
    "fwd_ret",
    "volatility_7d",
    "atr_pct",
    "volatility_1d",
]

KEEP_ALWAYS_REG = KEEP_ALWAYS_CLF + [
    "price_log",
    "price_rank",
    "price_vs_sma_ratio",
    "price_zscore",
    "target",
]


# 1. ВСПОМОГАТЕЛЬНОЕ


def load_dataset(path: str, label: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[{label}] не найден '{path}'. Сначала запустите FE."
        )
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "datetime"
    return df


def feature_list(
    df: pd.DataFrame,
    dataset_type: Literal["clf", "reg"] = "clf",
) -> list:
    non_feat = _NON_FEATURE_CLF if dataset_type == "clf" else _NON_FEATURE_REG
    return [c for c in df.columns if c not in non_feat]


def compute_feature_scores(
    df: pd.DataFrame,
    feats: list,
    target_col: str,
    dataset_type: Literal["clf", "reg"] = "clf",
) -> pd.DataFrame:
    """
    Считает скоринговые метрики признаков.

    dataset_type="clf": MI через mutual_info_classif (целочисленный таргет)
    dataset_type="reg": MI через mutual_info_regression (непрерывный fwd_ret)
    """
    rho = df[feats].corrwith(df[target_col], method="spearman").abs()
    r   = df[feats].corrwith(df[target_col], method="pearson").abs()

    x = df[feats].fillna(0.0).values
    y = df[target_col].values

    if dataset_type == "clf":
        mi_arr = mutual_info_classif(
            x, y.astype(int),
            discrete_features=False,
            random_state=MI_RANDOM_STATE,
        )
    else:
        mi_arr = mutual_info_regression(
            x, y.astype(float),
            discrete_features=False,
            random_state=MI_RANDOM_STATE,
        )

    mi = pd.Series(mi_arr, index=feats)

    scores = pd.DataFrame({
        "rho_abs": rho,
        "r_abs": r,
        "mi": mi,
    })

    def _norm(s: pd.Series) -> pd.Series:
        rng = s.max() - s.min()
        if rng == 0 or np.isnan(rng):
            return pd.Series(0.0, index=s.index)
        return (s - s.min()) / rng

    rho_n = _norm(scores["rho_abs"])
    mi_n  = _norm(scores["mi"])
    scores["score"] = SCORE_WEIGHT_RHO * rho_n + SCORE_WEIGHT_MI * mi_n
    scores = scores.sort_values("score", ascending=False)
    scores["rank"] = range(1, len(scores) + 1)
    return scores


def find_weak(scores: pd.DataFrame, keep_always: list) -> list:
    mask = (
        (scores["rho_abs"] < THRESHOLD_WEAK_RHO)
        & (scores["mi"]    < THRESHOLD_WEAK_MI)
    )
    return [f for f in scores.index[mask] if f not in keep_always]


def _connected_components(nodes: list, edges: list) -> list:
    parent = {n: n for n in nodes}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for a, b in edges:
        union(a, b)

    groups = defaultdict(list)
    for n in nodes:
        groups[find(n)].append(n)
    return list(groups.values())


def find_duplicates(
    df: pd.DataFrame,
    feats: list,
    scores: pd.DataFrame,
    already_drop: set,
    keep_always: list,
) -> tuple:
    candidates = [f for f in feats if f not in already_drop]
    fc = df[candidates].corr(method="pearson").abs()
    upper = fc.where(
        np.triu(np.ones(fc.shape), k=1).astype(bool)
    )

    pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feat_1", "level_1": "feat_2", 0: "r"})
        .query("r > @CORR_DUP_THRESHOLD")
        .sort_values("r", ascending=False)
        .reset_index(drop=True)
    )

    if pairs.empty:
        return [], pairs, []

    edges = list(zip(pairs["feat_1"], pairs["feat_2"]))
    components = _connected_components(candidates, edges)

    drop_list = []
    group_decisions = []

    for comp in components:
        if len(comp) < 2:
            continue
        protected_in_group = [f for f in comp if f in keep_always]
        if protected_in_group:
            winner = protected_in_group[0]
            losers = [f for f in comp if f not in keep_always]
        else:
            comp_scores = scores.loc[
                [f for f in comp if f in scores.index], "score"
            ]
            if comp_scores.empty:
                continue
            winner = comp_scores.idxmax()
            losers = [f for f in comp if f != winner]

        drop_list.extend(losers)
        group_decisions.append({
            "group_size": len(comp),
            "winner": winner,
            "winner_score": float(
                scores.loc[winner, "score"]
                if winner in scores.index else 0.0
            ),
            "losers": losers,
            "max_r_in_group": float(
                fc.loc[comp, comp]
                .where(~np.eye(len(comp), dtype=bool))
                .max().max()
            ),
        })

    seen = set()
    drop_list_unique = []
    for f in drop_list:
        if f not in seen and f not in already_drop:
            seen.add(f)
            drop_list_unique.append(f)

    return drop_list_unique, pairs, group_decisions


def signal_by_year(
    df: pd.DataFrame,
    label: str,
    target_col: str,
    dataset_type: Literal["clf", "reg"] = "clf",
) -> dict:
    print(f"\n  ── Статистика по годам [{label}] ──")
    tmp = df.copy()
    tmp["_yr"] = tmp.index.year
    by_year = {}

    for yr, g in tmp.groupby("_yr"):
        if dataset_type == "clf":
            rate = float((g[target_col] != 0).mean())
            up   = float((g[target_col] == 1).mean())
            dn   = float((g[target_col] == -1).mean())
            print(
                f"    {yr}: {len(g):5d} баров | "
                f"signal={rate:.3f} up={up:.3f} dn={dn:.3f}"
            )
            by_year[int(yr)] = {
                "n": int(len(g)),
                "signal_rate": rate,
                "up_rate": up,
                "down_rate": dn,
            }
        else:
            mean_ = float(g[target_col].mean())
            std_  = float(g[target_col].std())
            print(
                f"    {yr}: {len(g):5d} баров | "
                f"fwd_ret mean={mean_:.5f} std={std_:.5f}"
            )
            by_year[int(yr)] = {
                "n": int(len(g)),
                "fwd_ret_mean": mean_,
                "fwd_ret_std":  std_,
            }

    return by_year


# 2. ОСНОВНАЯ ФУНКЦИЯ ОБРАБОТКИ


def process(
    label: str,
    in_path: str,
    out_path: str,
    dataset_type: Literal["clf", "reg"] = "clf",
) -> dict | None:
    """
    Обрабатывает один датасет: скоринг → weak → dup → сохранение.

    dataset_type="clf": target=TARGET_CLF ("target"), MI = classif
    dataset_type="reg": target=TARGET_REG ("fwd_ret"), MI = regression
                        Ценовой контекст защищён от удаления.
    """
    print("\n" + "=" * 72)
    print(f"  ТАЙМФРЕЙМ {label} | тип={dataset_type.upper()}")
    print("=" * 72)

    df = load_dataset(in_path, label)

    if dataset_type == "clf":
        if TARGET_CLF not in df.columns:
            avail = [c for c in df.columns if "target" in c.lower()]
            raise ValueError(
                f"[{label}] нет '{TARGET_CLF}'. "
                f"Колонки с 'target': {avail}"
            )
        target_col = TARGET_CLF
        keep_always = KEEP_ALWAYS_CLF
    else:
        if TARGET_REG not in df.columns:
            raise ValueError(
                f"[{label}] нет '{TARGET_REG}' "
                f"(нужен для reg-датасета)"
            )
        target_col = TARGET_REG
        keep_always = KEEP_ALWAYS_REG

    feats = feature_list(df, dataset_type=dataset_type)

    n = len(df)
    if dataset_type == "clf":
        vc = df[target_col].value_counts().sort_index()
        dist = "  ".join(
            f"{int(k):+d}: {v} ({v / n * 100:.1f}%)"
            for k, v in vc.items()
        )
        print(f"  Target:  {dist}")
    else:
        print(
            f"  fwd_ret: mean={df[target_col].mean():.5f}  "
            f"std={df[target_col].std():.5f}  "
            f"q01={df[target_col].quantile(0.01):.4f}  "
            f"q99={df[target_col].quantile(0.99):.4f}"
        )
        if "target" in df.columns:
            vc_t = df["target"].value_counts().sort_index()
            print(f"  (3-class target в датасете): {vc_t.to_dict()}")

    print(f"  Файл:    {in_path}")
    print(f"  Строк:   {n:,} | признаков: {len(feats)}")
    print(f"  Период:  {df.index[0]} — {df.index[-1]}")

    year_stats = signal_by_year(df, label, target_col, dataset_type)

    print(
        f"\n  ── Скоринг признаков "
        f"({'MI_classif' if dataset_type == 'clf' else 'MI_regression'}) ──"
    )
    scores = compute_feature_scores(
        df, feats, target_col, dataset_type=dataset_type
    )
    print(f"\n  Топ-20 по score:")
    print(scores.head(20).round(5).to_string())

    drop_weak = find_weak(scores, keep_always)
    print(
        f"\n  ── Weak-фильтр "
        f"(|ρ|<{THRESHOLD_WEAK_RHO} И MI<{THRESHOLD_WEAK_MI}) ──"
    )
    print(f"  К удалению: {len(drop_weak)}")
    for f in sorted(drop_weak):
        print(
            f"    - {f:<42}  "
            f"|ρ|={scores.loc[f, 'rho_abs']:.5f}  "
            f"MI={scores.loc[f, 'mi']:.5f}"
        )

    #(низкая ρ, но высокий MI)
    saved_by_mi = [
        f for f in scores.index
        if (
            scores.loc[f, "rho_abs"] < THRESHOLD_WEAK_RHO
            and scores.loc[f, "mi"] >= THRESHOLD_WEAK_MI
            and f not in keep_always
        )
    ]
    if saved_by_mi:
        print(f"\n  ── Спасены по MI ({len(saved_by_mi)}) ──")
        for f in saved_by_mi:
            print(
                f"    + {f:<42}  "
                f"|ρ|={scores.loc[f, 'rho_abs']:.5f}  "
                f"MI={scores.loc[f, 'mi']:.5f}"
            )

    # Дедупликация
    drop_dup, pairs, group_decisions = find_duplicates(
        df, feats, scores, set(drop_weak), keep_always
    )

    print(
        f"\n  ── Дедупликация "
        f"(|r_pearson|>{CORR_DUP_THRESHOLD}) ──"
    )
    print(f"  Пар выше порога: {len(pairs)}")
    print(f"  Групп связности: {len(group_decisions)}")
    print(f"  К удалению: {len(drop_dup)}")
    for dec in group_decisions:
        print(
            f"    группа {dec['group_size']}, "
            f"max|r|={dec['max_r_in_group']:.3f}: "
            f"оставлен {dec['winner']} "
            f"(score={dec['winner_score']:.3f}), "
            f"удалены {dec['losers']}"
        )

    # NaN
    all_nan = [c for c in df.columns if df[c].isna().all()]
    if all_nan:
        print(f"\n  Удалено (100% NaN): {len(all_nan)}: {all_nan}")

    # Финальный набор
    drop_all = set(drop_weak) | set(drop_dup) | set(all_nan)
    present_meta = [
        c for c in keep_always
        if c in df.columns and c not in all_nan
    ]

    vol_required = ["volatility_1d", "volatility_7d", "atr_pct"]
    missing_vol = [c for c in vol_required if c not in present_meta]
    if missing_vol:
        print(f"\n  ⚠ vol-индикаторы отсутствуют: {missing_vol}")
    else:
        print(f"\n  ✓ vol-индикаторы защищены: {vol_required}")

    if dataset_type == "reg":
        price_ctx = [
            "price_log", "price_rank",
            "price_vs_sma_ratio", "price_zscore",
        ]
        missing_price = [c for c in price_ctx if c not in present_meta]
        if missing_price:
            print(f"  ⚠ ценовой контекст отсутствует: {missing_price}")
        else:
            print(f"  ✓ ценовой контекст защищён: {price_ctx}")

    final_feats = [
        f for f in feats
        if f not in drop_all and f not in present_meta
    ]

    if dataset_type == "clf":
        out_cols = present_meta + final_feats + [TARGET_CLF]
    else:
        extra_target = (
            ["target"] if "target" in df.columns
            and "target" not in present_meta else []
        )
        out_cols = present_meta + final_feats + extra_target


    seen_cols: set = set()
    out_cols_dedup = []
    for c in out_cols:
        if c in df.columns and c not in seen_cols:
            out_cols_dedup.append(c)
            seen_cols.add(c)
    out_cols = out_cols_dedup

    df_out = df[out_cols].copy()

    n_before = len(df_out)
    df_out.ffill(inplace=True)
    df_out.dropna(inplace=True)
    n_dropped = n_before - len(df_out)
    if n_dropped:
        print(
            f"  Удалено строк с NaN: {n_dropped} → "
            f"осталось {len(df_out):,}"
        )

    if len(df_out) == 0:
        print(f"  ❌ Датасет пуст после очистки — сохранение пропущено")
        return None

    df_out.to_csv(out_path, index=True, index_label="datetime")

    #Сохранение feature_scores
    scores_path = out_path.replace(
        "final_dataset_", "feature_scores_"
    )
    scores_out = scores.copy()
    scores_out["status"] = "kept"
    scores_out.loc[
        scores_out.index.isin(drop_weak), "status"
    ] = "dropped_weak"
    scores_out.loc[
        scores_out.index.isin(drop_dup), "status"
    ] = "dropped_duplicate"
    scores_out.loc[
        scores_out.index.isin(all_nan), "status"
    ] = "dropped_all_nan"
    scores_out.loc[
        scores_out.index.isin(present_meta), "status"
    ] = "meta"
    scores_out.to_csv(scores_path, index_label="feature")

    report_path = out_path.replace(
        "final_dataset_", "selection_report_"
    ).replace(".csv", ".json")

    vc_dict: dict[str, Any]
    if dataset_type == "clf":
        vc_d = df[target_col].value_counts().sort_index()
        vc_dict = {str(int(k)): int(v) for k, v in vc_d.items()}
    else:
        vc_dict = {
            "mean": float(df[target_col].mean()),
            "std":  float(df[target_col].std()),
        }

    report: dict[str, Any] = {
        "label": label,
        "dataset_type": dataset_type,
        "target_col": target_col,
        "input": in_path,
        "output": out_path,
        "params": {
            "threshold_weak_rho": THRESHOLD_WEAK_RHO,
            "threshold_weak_mi":  THRESHOLD_WEAK_MI,
            "corr_dup_threshold": CORR_DUP_THRESHOLD,
            "score_weight_rho":   SCORE_WEIGHT_RHO,
            "score_weight_mi":    SCORE_WEIGHT_MI,
        },
        "input_stats": {
            "n_rows":       int(n),
            "n_features":   int(len(feats)),
            "period_start": str(df.index[0]),
            "period_end":   str(df.index[-1]),
            "target_distribution": vc_dict,
            "by_year": year_stats,
        },
        "selection": {
            "dropped_weak":       sorted(drop_weak),
            "saved_by_mi":        sorted(saved_by_mi),
            "dropped_duplicate":  sorted(drop_dup),
            "dropped_all_nan":    sorted(all_nan),
            "group_decisions":    group_decisions,
            "kept_features":      final_feats,
            "meta_columns":       present_meta,
        },
        "output_stats": {
            "n_rows":           int(len(df_out)),
            "n_features_final": int(len(final_feats)),
            "n_meta":           int(len(present_meta)),
        },
    }
    with open(report_path, "w", encoding="utf-8") as fh:
        json.dump(report, fh, indent=2, ensure_ascii=False)

    print(f"\n  ── Итог [{label}] ──")
    print(f"  Исходных признаков:   {len(feats)}")
    print(f"  Удалено weak:         {len(drop_weak)}")
    print(f"  Спасено по MI:        {len(saved_by_mi)}")
    print(f"  Удалено дубликатов:   {len(drop_dup)}")
    print(f"  Удалено all-NaN:      {len(all_nan)}")
    print(f"  Финальных фич:        {len(final_feats)}")
    print(f"  Мета-колонок:         {len(present_meta)}")
    print(f"  Строк:                {len(df_out):,}")
    print(
        f"  Период:               "
        f"{df_out.index[0]} — {df_out.index[-1]}"
    )
    print(f"  Сохранено: {out_path}, {scores_path}, {report_path}")

    return {
        "label": label,
        "dataset_type": dataset_type,
        "n_input":    len(feats),
        "n_weak":     len(drop_weak),
        "n_saved_mi": len(saved_by_mi),
        "n_dup":      len(drop_dup),
        "n_nan":      len(all_nan),
        "n_final":    len(final_feats),
        "n_meta":     len(present_meta),
        "n_rows":     len(df_out),
        "feats":      final_feats,
    }



# 3. ЗАПУСК — 4 датасета

if __name__ == "__main__":

    TASKS = [
        ("1H-all",  INPUT_1H,      OUTPUT_1H,      "clf"),
        ("6H-all",  INPUT_6H,      OUTPUT_6H,      "clf"),
        ("1H-jump", INPUT_1H_JUMP, OUTPUT_1H_JUMP, "reg"),
        ("6H-jump", INPUT_6H_JUMP, OUTPUT_6H_JUMP, "reg"),
    ]

    results = {}
    for label, in_path, out_path, dtype in TASKS:
        if not os.path.exists(in_path):
            print(
                f"\n  ⚠ {in_path} не найден — "
                f"датасет {label} пропущен"
            )
            continue
        res = process(label, in_path, out_path, dataset_type=dtype)
        if res is not None:
            results[label] = res

    print("\n" + "=" * 80)
    print("  СРАВНЕНИЕ ДАТАСЕТОВ")
    print("=" * 80)
    hdr = f"  {'Метрика':<25}"
    for lbl in results:
        hdr += f"{lbl:>13}"
    print(hdr)
    print("  " + "-" * (25 + 13 * len(results)))

    metrics = [
        ("n_input",    "Исходных признаков"),
        ("n_weak",     "Удалено (weak)"),
        ("n_saved_mi", "Спасено по MI"),
        ("n_dup",      "Удалено (дубл.)"),
        ("n_nan",      "Удалено (NaN)"),
        ("n_final",    "Финальных фич"),
        ("n_meta",     "Мета-колонок"),
        ("n_rows",     "Строк"),
    ]
    for k, lbl in metrics:
        row = f"  {lbl:<25}"
        for res in results.values():
            v = res[k]
            row += f"{v:>13,}" if k == "n_rows" else f"{v:>13}"
        print(row)

    # Пересечение признаков clf-датасетов
    clf_results = {
        k: v for k, v in results.items()
        if v["dataset_type"] == "clf"
    }
    if len(clf_results) == 2:
        keys = list(clf_results)
        s1 = set(clf_results[keys[0]]["feats"])
        s2 = set(clf_results[keys[1]]["feats"])
        print(
            f"\n  [{keys[0]} vs {keys[1]}] "
            f"только в {keys[0]} ({len(s1 - s2)}): {sorted(s1 - s2)}"
        )
        print(
            f"  [{keys[0]} vs {keys[1]}] "
            f"только в {keys[1]} ({len(s2 - s1)}): {sorted(s2 - s1)}"
        )
        print(
            f"  [{keys[0]} vs {keys[1]}] "
            f"общих ({len(s1 & s2)}): {sorted(s1 & s2)}"
        )

    print("\n" + "=" * 80)
    print("  Готово. Выходные файлы:")
    for label, in_path, out_path, dtype in TASKS:
        if label in results:
            r = results[label]
            print(
                f"    {out_path}  "
                f"({r['n_final']} фич + {r['n_meta']} мета | "
                f"type={dtype})"
            )
    print("=" * 80)
    print(
        "  Следующий шаг: training_block_v8.py читает "
        "4 датасета параллельно."
    )


  ⚠ features_1h.csv не найден — датасет 1H-all пропущен

  ⚠ features_6h.csv не найден — датасет 6H-all пропущен

  ⚠ features_1h_jump.csv не найден — датасет 1H-jump пропущен

  ⚠ features_6h_jump.csv не найден — датасет 6H-jump пропущен

  СРАВНЕНИЕ ДАТАСЕТОВ
  Метрика                  
  -------------------------
  Исходных признаков       
  Удалено (weak)           
  Спасено по MI            
  Удалено (дубл.)          
  Удалено (NaN)            
  Финальных фич            
  Мета-колонок             
  Строк                    

  Готово. Выходные файлы:
  Следующий шаг: training_block_v8.py читает 4 датасета параллельно.


In [ ]:
"""

БЛОК 4 — Визуализация финального датасета


Что строит:

  1. Pearson correlation matrix — heatmap всех признаков.
  2. Spearman / Pearson корреляции с target — горизонтальный barplot.
  3. Распределение классов target — bar + динамика по времени.
  4. Распределения признаков по классам — violin для топ-N.
  5. Mutual Information с target — barplot (нелинейные зависимости).
  6. Динамика цены c_close с разметкой сигналов target.
  7. Дендрограмма иерархической кластеризации признаков
     (находит группы похожих признаков, которые блок 3 мог пропустить).

=============================================================================
"""

import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import linkage
from sklearn.feature_selection import mutual_info_classif


# КОНФИГ


INPUT_1H = "final_dataset_1h.csv"
INPUT_6H = "final_dataset_6h.csv"
OUTPUT_1H = "viz_report_1h.html"
OUTPUT_6H = "viz_report_6h.html"

TARGET_COL = "target"
META_COLS = ["c_close", "fwd_ret", "volatility_1d", "volatility_7d", "atr_pct"]

TOP_N_FEATURES = 15 # сколько топ-фич показывать в violin
HEATMAP_MAX_FEATURES = 60 # ограничение размера матрицы для читаемости
PRICE_SAMPLE_STEP = 1 # 1 = все бары, >1 = прореживание для скорости


# 1. ЗАГРУЗКА


def load_final(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[{label}] не найден '{path}'. Сначала запустите блок 3."
        )
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "datetime"
    if TARGET_COL not in df.columns:
        raise ValueError(f"[{label}] нет колонки '{TARGET_COL}'.")
    return df


def split_columns(df):
    """Возвращает (feature_cols, meta_present)."""
    meta_present = [c for c in META_COLS if c in df.columns]
    feats = [
        c for c in df.columns
        if c not in meta_present and c != TARGET_COL
    ]
    return feats, meta_present


# 2. ГРАФИКИ


def fig_corr_heatmap(df, feats, label):
    """Pearson correlation matrix признаков."""
    if len(feats) > HEATMAP_MAX_FEATURES:
        rho = (
            df[feats]
            .corrwith(df[TARGET_COL], method="spearman")
            .abs()
            .sort_values(ascending=False)
        )
        feats_plot = rho.head(HEATMAP_MAX_FEATURES).index.tolist()
        title_suffix = f" (топ-{HEATMAP_MAX_FEATURES} по |ρ_spearman|)"
    else:
        feats_plot = feats
        title_suffix = ""

    corr = df[feats_plot].corr(method="pearson")

    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.columns,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Pearson r"),
        hovertemplate="%{y} ↔ %{x}<br>r = %{z:.3f}<extra></extra>",
    ))
    fig.update_layout(
        title=f"[{label}] Pearson correlation matrix{title_suffix}",
        width=950,
        height=900,
        xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9), autorange="reversed"),
    )
    return fig


def fig_target_corr(df, feats, label):
    """Spearman + Pearson корреляции с target — горизонтальный barplot."""
    sp = df[feats].corrwith(df[TARGET_COL], method="spearman")
    pe = df[feats].corrwith(df[TARGET_COL], method="pearson")
    corr_df = pd.DataFrame({"spearman": sp, "pearson": pe})
    corr_df["abs_sp"] = corr_df["spearman"].abs()
    corr_df = corr_df.sort_values("abs_sp", ascending=True)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=corr_df.index,
        x=corr_df["spearman"],
        name="Spearman ρ",
        orientation="h",
        marker_color="steelblue",
        hovertemplate="%{y}<br>ρ = %{x:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        y=corr_df.index,
        x=corr_df["pearson"],
        name="Pearson r",
        orientation="h",
        marker_color="indianred",
        opacity=0.65,
        hovertemplate="%{y}<br>r = %{x:.4f}<extra></extra>",
    ))
    fig.update_layout(
        title=f"[{label}] Корреляции признаков с target",
        barmode="group",
        height=max(400, 18 * len(feats)),
        width=950,
        xaxis=dict(title="Корреляция", zeroline=True,
                   zerolinewidth=2, zerolinecolor="black"),
        yaxis=dict(tickfont=dict(size=9)),
        legend=dict(orientation="h", yanchor="bottom", y=1.01),
    )
    return fig


def fig_target_distribution(df, label):
    """Распределение классов target + signal rate по годам."""
    vc = df[TARGET_COL].value_counts().sort_index()
    n = len(df)

    yearly = df.copy()
    yearly["_yr"] = yearly.index.year
    sig_by_year = (
        yearly.groupby("_yr")[TARGET_COL]
        .agg(
            total="size",
            up=lambda x: (x == 1).sum(),
            down=lambda x: (x == -1).sum(),
            neutral=lambda x: (x == 0).sum(),
        )
    )
    sig_by_year["up_rate"] = sig_by_year["up"] / sig_by_year["total"]
    sig_by_year["down_rate"] = sig_by_year["down"] / sig_by_year["total"]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Общее распределение классов",
                        "Доля сигналов по годам"),
        column_widths=[0.4, 0.6],
    )

    colors = {-1: "#d62728", 0: "#7f7f7f", 1: "#2ca02c"}
    fig.add_trace(
        go.Bar(
            x=[f"{int(k):+d}" for k in vc.index],
            y=vc.values,
            marker_color=[colors[k] for k in vc.index],
            text=[f"{v}<br>({v / n * 100:.1f}%)" for v in vc.values],
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Bar(
            x=sig_by_year.index,
            y=sig_by_year["up_rate"],
            name="up (+1)",
            marker_color="#2ca02c",
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Bar(
            x=sig_by_year.index,
            y=sig_by_year["down_rate"],
            name="down (-1)",
            marker_color="#d62728",
        ),
        row=1, col=2,
    )

    fig.update_layout(
        title=f"[{label}] Распределение target",
        height=450,
        width=1100,
        barmode="stack",
        legend=dict(orientation="h", yanchor="bottom", y=1.05),
    )
    fig.update_yaxes(title_text="Количество баров", row=1, col=1)
    fig.update_yaxes(title_text="Доля баров", row=1, col=2)
    return fig


def fig_violin_top_features(df, feats, label):
    """Violin-распределения топ-N признаков в разрезе классов target."""
    rho = (
        df[feats]
        .corrwith(df[TARGET_COL], method="spearman")
        .abs()
        .sort_values(ascending=False)
    )
    top = rho.head(TOP_N_FEATURES).index.tolist()

    n_cols = 3
    n_rows = int(np.ceil(len(top) / n_cols))
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[
            f"{f}  (|ρ|={rho[f]:.3f})" for f in top
        ],
        vertical_spacing=0.06,
        horizontal_spacing=0.06,
    )

    colors = {-1: "#d62728", 0: "#7f7f7f", 1: "#2ca02c"}
    for i, feat in enumerate(top):
        r = i // n_cols + 1
        c = i % n_cols + 1
        for cls in sorted(df[TARGET_COL].unique()):
            mask = df[TARGET_COL] == cls
            # клиппинг по 1/99 перцентилям, чтобы хвосты не давили
            vals = df.loc[mask, feat]
            lo, hi = np.nanpercentile(vals, [1, 99])
            vals = vals.clip(lo, hi)
            fig.add_trace(
                go.Violin(
                    y=vals,
                    name=f"{int(cls):+d}",
                    legendgroup=f"{int(cls):+d}",
                    showlegend=(i == 0),
                    line_color=colors[cls],
                    fillcolor=colors[cls],
                    opacity=0.55,
                    box_visible=True,
                    meanline_visible=True,
                ),
                row=r, col=c,
            )

    fig.update_layout(
        title=(
            f"[{label}] Распределения топ-{TOP_N_FEATURES} признаков "
            f"по классам target"
        ),
        height=320 * n_rows,
        width=1200,
        violinmode="group",
    )
    for ann in fig["layout"]["annotations"]:
        ann["font"] = dict(size=10)
    return fig


def fig_mutual_information(df, feats, label):
    """Mutual Information с target — ловит нелинейные зависимости."""
    x = df[feats].values
    y = df[TARGET_COL].values.astype(int)

    valid = ~np.isnan(x).any(axis=1)
    if valid.sum() < len(x):
        x = x[valid]
        y = y[valid]

    mi = mutual_info_classif(
        x, y,
        discrete_features=False,
        random_state=42,
    )
    mi_df = (
        pd.Series(mi, index=feats, name="mi")
        .sort_values(ascending=True)
    )

    fig = go.Figure(go.Bar(
        y=mi_df.index,
        x=mi_df.values,
        orientation="h",
        marker=dict(
            color=mi_df.values,
            colorscale="Viridis",
            colorbar=dict(title="MI"),
        ),
        hovertemplate="%{y}<br>MI = %{x:.4f}<extra></extra>",
    ))
    fig.update_layout(
        title=(
            f"[{label}] Mutual Information с target "
            f"(нелинейная зависимость)"
        ),
        height=max(400, 18 * len(feats)),
        width=950,
        xaxis_title="Mutual Information",
        yaxis=dict(tickfont=dict(size=9)),
    )
    return fig


def fig_price_with_signals(df, label):
    """Цена c_close с разметкой сигналов target (±1)."""
    if "c_close" not in df.columns:
        return None

    sub = df.iloc[::PRICE_SAMPLE_STEP].copy()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub.index,
        y=sub["c_close"],
        mode="lines",
        name="BTC close",
        line=dict(color="#1f77b4", width=1),
        hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>",
    ))

    up_mask = sub[TARGET_COL] == 1
    dn_mask = sub[TARGET_COL] == -1

    fig.add_trace(go.Scatter(
        x=sub.index[up_mask],
        y=sub.loc[up_mask, "c_close"],
        mode="markers",
        name="target = +1",
        marker=dict(color="#2ca02c", size=4, opacity=0.6,
                    symbol="triangle-up"),
        hovertemplate="UP<br>%{x}<br>$%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=sub.index[dn_mask],
        y=sub.loc[dn_mask, "c_close"],
        mode="markers",
        name="target = -1",
        marker=dict(color="#d62728", size=4, opacity=0.6,
                    symbol="triangle-down"),
        hovertemplate="DOWN<br>%{x}<br>$%{y:,.0f}<extra></extra>",
    ))

    fig.update_layout(
        title=f"[{label}] BTC цена с разметкой сигналов target",
        height=500,
        width=1200,
        xaxis_title="Время",
        yaxis_title="Цена, USD",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    return fig


def fig_feature_dendrogram(df, feats, label):
    """
    Иерархическая кластеризация признаков по 1 - |Pearson r|.
    Помогает увидеть кластеры похожих признаков, которые блок 3 мог
    не отсечь (он работает только по парам).
    """
    if len(feats) < 3:
        return None

    if len(feats) > HEATMAP_MAX_FEATURES:
        rho = (
            df[feats]
            .corrwith(df[TARGET_COL], method="spearman")
            .abs()
            .sort_values(ascending=False)
        )
        feats_plot = rho.head(HEATMAP_MAX_FEATURES).index.tolist()
        title_suffix = f" (топ-{HEATMAP_MAX_FEATURES})"
    else:
        feats_plot = feats
        title_suffix = ""

    corr = df[feats_plot].corr(method="pearson").abs()
    dist = 1 - corr
    n = len(feats_plot)
    cond = dist.values[np.triu_indices(n, k=1)]

    z = linkage(cond, method="average")

    fig = ff.create_dendrogram(
        corr.values,
        labels=feats_plot,
        linkagefun=lambda _: z,
        orientation="left",
        color_threshold=0.3,
    )
    fig.update_layout(
        title=(
            f"[{label}] Дендрограмма признаков "
            f"(дистанция = 1 − |Pearson r|){title_suffix}"
        ),
        width=1100,
        height=max(500, 18 * n),
        xaxis_title="Дистанция",
        yaxis=dict(tickfont=dict(size=9)),
    )
    return fig


# 3. СБОРКА HTML


def build_report(label, in_path, out_path):
    print("\n" + "=" * 72)
    print(f"  ТАЙМФРЕЙМ {label}")
    print("=" * 72)

    df = load_final(in_path, label)
    feats, meta = split_columns(df)

    print(f"  Файл:    {in_path}")
    print(f"  Строк:   {len(df):,}")
    print(f"  Фич:     {len(feats)}")
    print(f"  Мета:    {meta}")
    print(f"  Период:  {df.index[0]} — {df.index[-1]}")

    figures = []

    print("  → распределение target")
    figures.append(fig_target_distribution(df, label))

    print("  → цена с сигналами")
    f = fig_price_with_signals(df, label)
    if f is not None:
        figures.append(f)

    print("  → корреляции с target")
    figures.append(fig_target_corr(df, feats, label))

    print("  → mutual information")
    figures.append(fig_mutual_information(df, feats, label))

    print("  → heatmap Pearson")
    figures.append(fig_corr_heatmap(df, feats, label))

    print("  → дендрограмма")
    f = fig_feature_dendrogram(df, feats, label)
    if f is not None:
        figures.append(f)

    print("  → violin топ-фич")
    figures.append(fig_violin_top_features(df, feats, label))

    parts = [
        "<!doctype html><html lang='ru'><head><meta charset='utf-8'>",
        f"<title>BTC features report [{label}]</title>",
        "<style>",
        "  body { font-family: -apple-system, system-ui, sans-serif; ",
        "         max-width: 1300px; margin: 24px auto; padding: 0 16px; ",
        "         color: #222; background: #fafafa; }",
        "  h1 { margin-bottom: 4px; }",
        "  .meta { color: #666; margin-bottom: 24px; font-size: 14px; }",
        "  .fig { background: #fff; border: 1px solid #e3e3e3; ",
        "         border-radius: 8px; margin: 18px 0; padding: 12px; ",
        "         box-shadow: 0 1px 3px rgba(0,0,0,0.04); }",
        "</style></head><body>",
        f"<h1>BTC features report — {label}</h1>",
        f"<div class='meta'>",
        f"  Источник: <code>{in_path}</code> &nbsp;|&nbsp; ",
        f"  строк: {len(df):,} &nbsp;|&nbsp; ",
        f"  признаков: {len(feats)} &nbsp;|&nbsp; ",
        f"  период: {df.index[0].date()} — {df.index[-1].date()}",
        f"</div>",
    ]

    for i, fig in enumerate(figures):
        parts.append("<div class='fig'>")
        parts.append(fig.to_html(
            full_html=False,
            include_plotlyjs="cdn" if i == 0 else False,
        ))
        parts.append("</div>")

    parts.append("</body></html>")

    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(parts))

    print(f"\n  ✓ Сохранено: {out_path}  ({len(figures)} графиков)")
    return out_path



# 4. ЗАПУСК


if __name__ == "__main__":
    outputs = []
    for label, in_path, out_path in [
        ("1H", INPUT_1H, OUTPUT_1H),
        ("6H", INPUT_6H, OUTPUT_6H),
    ]:
        if not os.path.exists(in_path):
            print(f"\n  ⚠ {in_path} не найден — таймфрейм {label} пропущен")
            continue
        outputs.append(build_report(label, in_path, out_path))

    print("\n" + "=" * 72)
    print("  Готово.")
    for p in outputs:
        print(f"    {p}")
    print("=" * 72)

In [ ]:
"""
БЛОК ТЮНИНГА — гиперпараметры gate + reg_all + reg_jump

Парный с training_block_v8.py.

"""

import json
import os
import time
import warnings
from datetime import datetime
from typing import Any, Callable

import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostClassifier

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    _HAS_LGBM = True
except ImportError:
    _HAS_LGBM = False

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)




BUDGET: str = "light"

BUDGET_CFG: dict[str, dict[str, int]] = {
    "light":    {"n_trials": 40,  "timeout_min": 8},
    "standard": {"n_trials": 80,  "timeout_min": 18},
    "heavy":    {"n_trials": 150, "timeout_min": 35},
}

MODELS_TO_TUNE_GATE: list[str] | None = None
MODELS_TO_TUNE_REG:  list[str] | None = None

ALL_GATE_MODELS: list[str] = ["xgb", "lgbm", "cat"]
ALL_REG_MODELS:  list[str] = [
    "xgb_huber", "lgbm_huber", "hgb_mae", "elasticnet",
]

GATE_OBJECTIVE: str = "auc"
DIR_OBJECTIVE:  str = "mae"

TRAIN_TARGET: str = "t_static"

DATA_1H:      str = "final_dataset_1h.csv"
DATA_6H:      str = "final_dataset_6h.csv"
DATA_1H_JUMP: str = "final_dataset_1h_jump.csv"
DATA_6H_JUMP: str = "final_dataset_6h_jump.csv"

STATIC_THRESH_1H: float = 0.008
STATIC_THRESH_6H: float = 0.015

N_WF_SPLITS:  int = 5
EMBARGO_1H:   int = 1
EMBARGO_6H:   int = 6

OUTPUT_FILE: str = "best_params_reg.json"

RANDOM_STATE: int = 42
FWD_COL:      str = "fwd_ret"

_LEAK_PREFIX: tuple[str, ...] = (
    "fwd_ret", "target", "t_static", "t_dynamic",
)
_NON_FEATURE: set[str] = {"c_close"}

_PRICE_CONTEXT_COLS: set[str] = {
    "price_log",
    "price_rank",
    "price_vs_sma_ratio",
    "price_zscore",
}

# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

def _resolve_path(path: str, label: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{label}] не найден '{path}'")
    print(f"  [{label}] вход: {path}")
    return path


def build_data(
    df: pd.DataFrame,
    static_thresh: float,
) -> pd.DataFrame:
    """
    Добавляет t_static (нужен для gate-таргета и отчётности).
    fwd_ret должен присутствовать в df.
    """
    if FWD_COL not in df.columns:
        raise KeyError(f"Нет колонки '{FWD_COL}'")
    df = df.dropna(subset=[FWD_COL]).copy()
    fwd = df[FWD_COL]
    df["t_static"] = np.where(
        fwd > static_thresh, 1,
        np.where(fwd < -static_thresh, -1, 0),
    )
    return df


def split_features(df: pd.DataFrame) -> list[str]:
    """Признаки для gate и reg_all (без leakage и мета-колонок)."""
    return [
        c for c in df.columns
        if c not in _NON_FEATURE
        and not c.startswith(_LEAK_PREFIX)
    ]


def split_features_jump(df: pd.DataFrame) -> list[str]:
    """
    Признаки для reg_jump.
    Ценовой контекст (price_*) уже присутствует в датасете
    от Блока 3, поэтому используем ту же логику, что и split_features.
    """
    return [
        c for c in df.columns
        if c not in _NON_FEATURE
        and not c.startswith(_LEAK_PREFIX)
    ]


def make_gate_xy(
    df: pd.DataFrame,
    feat_cols: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """Gate: все бары, y=1 если |fwd_ret| > static_thresh."""
    X = df[feat_cols].copy()
    y = (df["t_static"] != 0).astype(int)
    ok = X.notna().all(axis=1)
    return X.loc[ok], y.loc[ok]


def make_reg_all_xy(
    df: pd.DataFrame,
    feat_cols: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """REG-ALL: все бары, y=fwd_ret."""
    X = df[feat_cols].copy()
    y = df[FWD_COL].copy()
    ok = X.notna().all(axis=1) & y.notna()
    return X.loc[ok], y.loc[ok]


def make_reg_jump_xy(
    df_jump: pd.DataFrame,
    feat_cols_jump: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """
    REG-JUMP: только строки скачков (уже отфильтрованы в Блоке 3).
    y=fwd_ret. Признаки включают price_*.
    """
    X = df_jump[feat_cols_jump].copy()
    y = df_jump[FWD_COL].copy()
    ok = X.notna().all(axis=1) & y.notna()
    return X.loc[ok], y.loc[ok]

# 2. PURGED WALK-FORWARD С EMBARGO

def purged_embargo_splits(
    n: int,
    n_splits: int,
    embargo: int,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """
    Возвращает список (train_idx, val_idx) с purge-gap = embargo.
    Минимальный train = 50, минимальный val = 10.
    """
    fold = n // (n_splits + 1)
    out: list[tuple[np.ndarray, np.ndarray]] = []
    for k in range(1, n_splits + 1):
        cut    = fold * k
        tr_end = cut - embargo
        va_end = min(cut + fold, n)
        if tr_end <= 50 or va_end <= cut + 10:
            continue
        out.append((np.arange(0, tr_end), np.arange(cut, va_end)))
    return out


# 3. МЕТРИКИ

def _reg_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    """MAE / RMSE / sign_acc на ненулевых барах."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask = np.abs(y_true) > 1e-9
    sign_acc = (
        float(np.mean(
            np.sign(y_pred[mask]) == np.sign(y_true[mask])
        ))
        if mask.sum() > 0 else float("nan")
    )
    return {"mae": mae, "rmse": rmse, "sign_acc": sign_acc}


def _clf_metrics(
    y_true: np.ndarray,
    y_proba: np.ndarray,
) -> dict[str, float]:
    """AUC для бинарного gate-классификатора."""
    n_cls = len(np.unique(y_true))
    if n_cls < 2:
        return {"auc": float("nan")}
    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = float("nan")
    return {"auc": auc}


def cv_score_reg(
    estimator_fn: Callable,
    X: pd.DataFrame,
    y: pd.Series,
    splits: list[tuple[np.ndarray, np.ndarray]],
    target_metric: str,
) -> tuple[dict[str, float], float]:
    """WF-OOS CV для регрессора. Возвращает (avg_metrics, score)."""
    accum: dict[str, list[float]] = {
        "mae": [], "rmse": [], "sign_acc": [],
    }
    n_X = len(X)
    for tr, va in splits:
        if tr.max() >= n_X or va.max() >= n_X or len(va) == 0:
            raise ValueError(
                f"Сплит вне X: n_X={n_X}, "
                f"tr.max={tr.max()}, va.max={va.max()}"
            )
        m = estimator_fn()
        m.fit(X.iloc[tr], y.iloc[tr])
        pr = m.predict(X.iloc[va])
        d  = _reg_metrics(
            y.iloc[va].to_numpy(), np.asarray(pr)
        )
        for k in accum:
            v = d[k]
            if v == v:  # не NaN
                accum[k].append(v)

    avg = {
        k: (float(np.mean(v)) if v else float("nan"))
        for k, v in accum.items()
    }
    return avg, avg.get(target_metric, float("nan"))


def cv_score_clf(
    estimator_fn: Callable,
    X: pd.DataFrame,
    y: pd.Series,
    splits: list[tuple[np.ndarray, np.ndarray]],
    target_metric: str,
) -> tuple[dict[str, float], float]:
    """WF-OOS CV для бинарного классификатора. Возвращает (avg_metrics, score)."""
    accum: dict[str, list[float]] = {"auc": []}
    n_X = len(X)
    for tr, va in splits:
        if tr.max() >= n_X or va.max() >= n_X or len(va) == 0:
            raise ValueError(
                f"Сплит вне X: n_X={n_X}, "
                f"tr.max={tr.max()}, va.max={va.max()}"
            )
        ytr = y.iloc[tr]
        if ytr.nunique() < 2:
            continue
        m = estimator_fn()
        m.fit(X.iloc[tr], ytr)
        pr = m.predict_proba(X.iloc[va])[:, 1]
        d  = _clf_metrics(
            y.iloc[va].to_numpy(), np.asarray(pr)
        )
        for k in accum:
            v = d[k]
            if v == v:
                accum[k].append(v)

    avg = {
        k: (float(np.mean(v)) if v else float("nan"))
        for k, v in accum.items()
    }
    return avg, avg.get(target_metric, float("nan"))

# 4. СЕТКИ ГИПЕРПАРАМЕТРОВ

def suggest_gate_xgb(
    trial: optuna.Trial,
    spw: float,
) -> dict[str, Any]:
    return {
        "objective":         "binary:logistic",
        "eval_metric":       "logloss",
        "tree_method":       "hist",
        "n_jobs":            -1,
        "verbosity":         0,
        "random_state":      RANDOM_STATE,
        "scale_pos_weight":  spw,
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 700),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 9),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 20),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-4, 10.0, log=True),
    }


def suggest_gate_lgbm(
    trial: optuna.Trial,
) -> dict[str, Any]:
    return {
        "objective":     "binary",
        "is_unbalance":  True,
        "n_jobs":        -1,
        "verbose":       -1,
        "random_state":  RANDOM_STATE,
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 700),
        "num_leaves": trial.suggest_int(
            "num_leaves", 8, 128),
        "max_depth": trial.suggest_int(
            "max_depth", -1, 12),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf", 5, 100),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int(
            "subsample_freq", 1, 10),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-4, 10.0, log=True),
    }


def suggest_gate_cat(
    trial: optuna.Trial,
    spw: float,
) -> dict[str, Any]:
    return {
        "loss_function":    "Logloss",
        "random_state":     RANDOM_STATE,
        "verbose":          0,
        "scale_pos_weight": spw,
        "iterations": trial.suggest_int(
            "iterations", 100, 700),
        "depth": trial.suggest_int(
            "depth", 3, 9),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-3, 20.0, log=True),
        "border_count": trial.suggest_int(
            "border_count", 32, 255),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 10.0),
    }


def suggest_xgb_huber(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "objective":    "reg:pseudohubererror",
        "tree_method":  "hist",
        "n_jobs":       -1,
        "verbosity":    0,
        "random_state": RANDOM_STATE,
        "huber_slope": trial.suggest_float(
            "huber_slope", 1e-3, 5e-2, log=True),
        "n_estimators": trial.suggest_int(
            "n_estimators", 150, 800),
        "max_depth": trial.suggest_int(
            "max_depth", 3, 9),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 30),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-4, 10.0, log=True),
    }


def suggest_lgbm_huber(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "objective":    "huber",
        "n_jobs":       -1,
        "verbose":      -1,
        "random_state": RANDOM_STATE,
        "alpha": trial.suggest_float(
            "alpha", 1e-3, 5e-2, log=True),
        "n_estimators": trial.suggest_int(
            "n_estimators", 150, 800),
        "num_leaves": trial.suggest_int(
            "num_leaves", 8, 128),
        "max_depth": trial.suggest_int(
            "max_depth", -1, 12),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf", 5, 100),
        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int(
            "subsample_freq", 1, 10),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-4, 10.0, log=True),
        "min_gain_to_split": trial.suggest_float(
            "min_gain_to_split", 0.0, 1.0),
    }


def suggest_hgb_mae(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "loss":         "absolute_error",
        "random_state": RANDOM_STATE,
        "max_iter": trial.suggest_int(
            "max_iter", 150, 800),
        "max_depth": trial.suggest_categorical(
            "max_depth", [3, 4, 5, 6, 8, 10, None]),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.15, log=True),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 10, 150),
        "l2_regularization": trial.suggest_float(
            "l2_regularization", 1e-4, 10.0, log=True),
        "max_leaf_nodes": trial.suggest_int(
            "max_leaf_nodes", 15, 127),
    }


def suggest_elasticnet(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "model__alpha": trial.suggest_float(
            "model__alpha", 1e-5, 1.0, log=True),
        "model__l1_ratio": trial.suggest_float(
            "model__l1_ratio", 0.05, 0.95),
        "model__max_iter":      5000,
        "model__random_state":  RANDOM_STATE,
    }

# 5. ФАБРИКИ ОЦЕНЩИКОВ

def make_gate_estimator(
    model_name: str,
    params: dict[str, Any],
) -> Any:
    """Создаёт gate-классификатор по имени и параметрам."""
    if model_name == "xgb":
        return XGBClassifier(**params)
    if model_name == "lgbm":
        if not _HAS_LGBM:
            raise RuntimeError("lightgbm не установлен")
        return LGBMClassifier(**params)
    if model_name == "cat":
        return CatBoostClassifier(**params)
    raise ValueError(f"Неизвестная gate-модель: '{model_name}'")


def make_reg_estimator(
    model_name: str,
    params: dict[str, Any],
) -> Any:
    """Создаёт регрессор по имени и параметрам."""
    if model_name == "xgb_huber":
        return XGBRegressor(**params)
    if model_name == "lgbm_huber":
        if not _HAS_LGBM:
            raise RuntimeError("lightgbm не установлен")
        return LGBMRegressor(**params)
    if model_name == "hgb_mae":
        return HistGradientBoostingRegressor(**params)
    if model_name == "elasticnet":
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet()),
        ])
        return pipe.set_params(**params)
    raise ValueError(f"Неизвестная reg-модель: '{model_name}'")


_REG_SUGGEST_FN: dict[str, Callable] = {
    "xgb_huber":  suggest_xgb_huber,
    "lgbm_huber": suggest_lgbm_huber,
    "hgb_mae":    suggest_hgb_mae,
    "elasticnet": suggest_elasticnet,
}


# 6. OPTUNA STUDY
def _run_study(
    objective_fn: Callable[[optuna.Trial], float],
    direction: str,
    n_trials: int,
    timeout_sec: float,
    study_name: str,
) -> optuna.Study:
    """Создаёт и запускает Optuna study. Возвращает study."""
    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_STATE, n_startup_trials=10,
    )
    pruner = optuna.pruners.MedianPruner(n_startup_trials=10)
    study  = optuna.create_study(
        direction=direction,
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
    )
    study.optimize(
        objective_fn,
        n_trials=n_trials,
        timeout=timeout_sec,
        show_progress_bar=False,
    )
    return study


def _study_result(
    study: optuna.Study,
    model_name: str,
    target_metric: str,
    elapsed: float,
) -> dict[str, Any]:
    """Упаковывает результат study в стандартный словарь."""
    completed = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]
    if not completed:
        return {
            "model":              model_name,
            "target_metric":      target_metric,
            "best_score":         float("nan"),
            "best_params":        {},
            "all_metrics_at_best": {},
            "n_trials":           len(study.trials),
            "elapsed_sec":        round(elapsed, 1),
            "warning":            "no completed trials",
        }
    best = study.best_trial
    return {
        "model":               model_name,
        "target_metric":       target_metric,
        "best_score":          float(best.value),
        "best_params":         dict(best.params),
        "all_metrics_at_best": dict(best.user_attrs),
        "n_trials":            len(study.trials),
        "elapsed_sec":         round(elapsed, 1),
    }

# 7. ТЮНИНГ GATE-КЛАССИФИКАТОРА

def tune_gate_one(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    splits: list[tuple[np.ndarray, np.ndarray]],
    n_trials: int,
    timeout_sec: float,
    study_name: str,
) -> dict[str, Any]:
    """
    Optuna study для одного gate-классификатора.
    Метрика: AUC (maximize).
    """
    spw = float((y == 0).sum()) / max(float((y == 1).sum()), 1.0)

    def objective(trial: optuna.Trial) -> float:
        if model_name == "xgb":
            params = suggest_gate_xgb(trial, spw)
        elif model_name == "lgbm":
            params = suggest_gate_lgbm(trial)
        elif model_name == "cat":
            params = suggest_gate_cat(trial, spw)
        else:
            raise optuna.TrialPruned(
                f"Неизвестная gate-модель: {model_name}"
            )

        try:
            avg, score = cv_score_clf(
                lambda: make_gate_estimator(model_name, params),
                X, y, splits, GATE_OBJECTIVE,
            )
        except Exception as exc:
            raise optuna.TrialPruned(f"fit failed: {exc}")

        for k, v in avg.items():
            trial.set_user_attr(k, v)
        return score if score == score else -1.0

    t0    = time.time()
    study = _run_study(
        objective, "maximize", n_trials, timeout_sec, study_name,
    )
    return _study_result(
        study, model_name, GATE_OBJECTIVE, time.time() - t0,
    )

# 8. ТЮНИНГ РЕГРЕССОРА (reg_all и reg_jump)

def tune_reg_one(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    splits: list[tuple[np.ndarray, np.ndarray]],
    n_trials: int,
    timeout_sec: float,
    study_name: str,
) -> dict[str, Any]:
    """
    Optuna study для одного регрессора (reg_all или reg_jump).
    Метрика: MAE (minimize).
    """
    suggest = _REG_SUGGEST_FN[model_name]

    def objective(trial: optuna.Trial) -> float:
        params = suggest(trial)
        try:
            avg, score = cv_score_reg(
                lambda: make_reg_estimator(model_name, params),
                X, y, splits, DIR_OBJECTIVE,
            )
        except Exception as exc:
            raise optuna.TrialPruned(f"fit failed: {exc}")

        for k, v in avg.items():
            trial.set_user_attr(k, v)
        return score if score == score else 1e9

    t0    = time.time()
    study = _run_study(
        objective, "minimize", n_trials, timeout_sec, study_name,
    )
    return _study_result(
        study, model_name, DIR_OBJECTIVE, time.time() - t0,
    )

# 9. ОРКЕСТРАЦИЯ — ОДИН ТАЙМФРЕЙМ


def tune_timeframe(
    name: str,
    path_all: str,
    path_jump: str,
    static_thresh: float,
    embargo: int,
    budget: dict[str, int],
) -> dict[str, Any]:
    """
    Полный цикл тюнинга для одного ТФ.

    Запускает три группы studies:
      gate     — на all-bars датасете
      dir      — на all-bars датасете (REG-ALL)
      dir_jump — на jump датасете    (REG-JUMP)

    Возвращает структуру, совместимую с training_block_v8:
      {TRAIN_TARGET: {'gate': {...}, 'dir': {...}, 'dir_jump': {...}}}
    """
    print("\n" + "=" * 72)
    print(
        f"  ТАЙМФРЕЙМ {name.upper()} — БЮДЖЕТ '{BUDGET}' | "
        f"embargo={embargo}"
    )
    print("=" * 72)

    n_trials    = budget["n_trials"]
    timeout_sec = budget["timeout_min"] * 60

    df_all = pd.read_csv(path_all, index_col=0, parse_dates=True)
    df_all = build_data(df_all, static_thresh)
    feat_cols = split_features(df_all)

    print(
        f"  [ALL]  строк={len(df_all)} | "
        f"признаков={len(feat_cols)}"
    )
    print(
        f"  fwd_ret: mean={df_all[FWD_COL].mean():.5f} "
        f"std={df_all[FWD_COL].std():.5f} "
        f"q01={df_all[FWD_COL].quantile(0.01):.4f} "
        f"q99={df_all[FWD_COL].quantile(0.99):.4f}"
    )
    vc_all = df_all["t_static"].value_counts().sort_index().to_dict()
    print(f"  t_static: {vc_all}")

    Xg, yg = make_gate_xy(df_all, feat_cols)
    Xr, yr = make_reg_all_xy(df_all, feat_cols)

    splits_all = purged_embargo_splits(len(Xg), N_WF_SPLITS, embargo)
    if not splits_all:
        raise RuntimeError(
            f"[{name}/all] Не построились сплиты "
            f"на {len(Xg)} строк"
        )

    df_jump = pd.read_csv(path_jump, index_col=0, parse_dates=True)
    df_jump = build_data(df_jump, static_thresh)
    feat_cols_jump = split_features_jump(df_jump)

    present_price_ctx = [
        c for c in _PRICE_CONTEXT_COLS
        if c in df_jump.columns
    ]
    feat_cols_jump = list(dict.fromkeys(
        feat_cols_jump + present_price_ctx
    ))

    print(
        f"  [JUMP] строк={len(df_jump)} | "
        f"признаков={len(feat_cols_jump)} "
        f"(price_ctx={len(present_price_ctx)})"
    )
    vc_jump = (
        df_jump["t_static"].value_counts().sort_index().to_dict()
    )
    print(f"  [JUMP] t_static: {vc_jump}")

    Xrj, yrj = make_reg_jump_xy(df_jump, feat_cols_jump)

    splits_jump = purged_embargo_splits(
        len(Xrj), N_WF_SPLITS, embargo,
    )
    if not splits_jump:
        print(
            f"  ⚠ [{name}/jump] Нет валидных сплитов "
            f"({len(Xrj)} строк) — reg_jump пропущен"
        )
        splits_jump = []

    print(f"\n  ── GATE (AUC, maximize) ──")
    gate_cands = (
        ALL_GATE_MODELS if MODELS_TO_TUNE_GATE is None
        else [
            m for m in ALL_GATE_MODELS
            if m in MODELS_TO_TUNE_GATE
        ]
    )
    gate_cands = [
        m for m in gate_cands
        if m != "lgbm" or _HAS_LGBM
    ]

    splits_gate = purged_embargo_splits(
        len(Xg), N_WF_SPLITS, embargo,
    )

    gate_out: dict[str, Any] = {}
    for mname in gate_cands:
        sn  = f"{name}_gate_{mname}"
        res = tune_gate_one(
            mname, Xg, yg, splits_gate,
            n_trials, timeout_sec, sn,
        )
        am = res["all_metrics_at_best"]
        print(
            f"    {mname:8}: "
            f"AUC={am.get('auc', float('nan')):.4f} | "
            f"trials={res['n_trials']} | "
            f"{res['elapsed_sec']:.0f}s"
        )
        gate_out[mname] = res

    print(f"\n  ── REG-ALL (MAE, minimize, все бары) ──")
    reg_cands = (
        ALL_REG_MODELS if MODELS_TO_TUNE_REG is None
        else [
            m for m in ALL_REG_MODELS
            if m in MODELS_TO_TUNE_REG
        ]
    )
    reg_cands = [
        m for m in reg_cands
        if m != "lgbm_huber" or _HAS_LGBM
    ]

    splits_reg_all = purged_embargo_splits(
        len(Xr), N_WF_SPLITS, embargo,
    )

    dir_out: dict[str, Any] = {}
    for mname in reg_cands:
        sn  = f"{name}_dir_{mname}"
        res = tune_reg_one(
            mname, Xr, yr, splits_reg_all,
            n_trials, timeout_sec, sn,
        )
        am = res["all_metrics_at_best"]
        print(
            f"    {mname:12}: "
            f"MAE={am.get('mae', float('nan')):.5f} "
            f"RMSE={am.get('rmse', float('nan')):.5f} "
            f"signA={am.get('sign_acc', float('nan')):.3f} | "
            f"trials={res['n_trials']} | "
            f"{res['elapsed_sec']:.0f}s"
        )
        dir_out[mname] = res

    print(f"\n  ── REG-JUMP (MAE, minimize, только скачки) ──")
    dir_jump_out: dict[str, Any] = {}

    if splits_jump:
        for mname in reg_cands:
            sn  = f"{name}_dir_jump_{mname}"
            res = tune_reg_one(
                mname, Xrj, yrj, splits_jump,
                n_trials, timeout_sec, sn,
            )
            am = res["all_metrics_at_best"]
            print(
                f"    {mname:12}: "
                f"MAE={am.get('mae', float('nan')):.5f} "
                f"RMSE={am.get('rmse', float('nan')):.5f} "
                f"signA={am.get('sign_acc', float('nan')):.3f} | "
                f"trials={res['n_trials']} | "
                f"{res['elapsed_sec']:.0f}s"
            )
            dir_jump_out[mname] = res
    else:
        print("  ⚠ REG-JUMP пропущен (недостаточно данных)")

    return {
        TRAIN_TARGET: {
            "gate":     gate_out,
            "dir":      dir_out,
            "dir_jump": dir_jump_out,
        }
    }


# 10. ЗАПУСК

if __name__ == "__main__":
    if not _HAS_LGBM:
        print(
            "  ⚠ lightgbm не установлен — "
            "lgbm_huber / lgbm будут пропущены"
        )

    budget = BUDGET_CFG[BUDGET]

    n_studies_per_tf = (
        len(ALL_GATE_MODELS)
        + len(ALL_REG_MODELS) * 2     # dir + dir_jump
    )
    total_studies = 2 * n_studies_per_tf

    print(f"\n{'#' * 72}")
    print(f"#  ТЮНИНГ v8 — gate + reg_all + reg_jump")
    print(f"#  Бюджет: '{BUDGET}' | "
          f"trials/study: {budget['n_trials']} | "
          f"timeout: {budget['timeout_min']} мин")
    print(f"#  Gate-модели:  {ALL_GATE_MODELS}")
    print(f"#  Reg-модели:   {ALL_REG_MODELS}")
    print(f"#  Studies всего: {total_studies} "
          f"(2 ТФ × {n_studies_per_tf})")
    print(f"#  Старт: {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"{'#' * 72}")

    _resolve_path(DATA_1H,      "1H-all")
    _resolve_path(DATA_6H,      "6H-all")
    _resolve_path(DATA_1H_JUMP, "1H-jump")
    _resolve_path(DATA_6H_JUMP, "6H-jump")

    t_start = time.time()
    result: dict[str, Any] = {
        "1h": tune_timeframe(
            "1h",
            path_all=DATA_1H,
            path_jump=DATA_1H_JUMP,
            static_thresh=STATIC_THRESH_1H,
            embargo=EMBARGO_1H,
            budget=budget,
        ),
        "6h": tune_timeframe(
            "6h",
            path_all=DATA_6H,
            path_jump=DATA_6H_JUMP,
            static_thresh=STATIC_THRESH_6H,
            embargo=EMBARGO_6H,
            budget=budget,
        ),
    }
    total_min = (time.time() - t_start) / 60

    merged:   dict[str, Any] = {}
    old_meta: dict[str, Any] = {}

    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE) as f:
                old = json.load(f)
            merged   = old.get("results", {}) or {}
            old_meta = old.get("_meta", {}) or {}
            print(
                f"\n  ↻ Найден старый {OUTPUT_FILE} от "
                f"{old_meta.get('created', '?')[:16]} — сливаем"
            )
        except Exception as exc:
            print(
                f"  ⚠ {OUTPUT_FILE} нечитаем ({exc}) — перезапись"
            )
            merged = {}

    # Обновляем только секции текущего запуска;
    # остальные разделы старого JSON сохраняются.
    for tf, by_t in result.items():
        merged.setdefault(tf, {})
        for tcol, by_s in by_t.items():
            merged[tf].setdefault(tcol, {})
            for stage, by_m in by_s.items():
                merged[tf][tcol][stage] = by_m

    payload: dict[str, Any] = {
        "_meta": {
            "created":            datetime.now().isoformat(
                timespec="seconds"
            ),
            "budget":             BUDGET,
            "n_trials_per_combo": budget["n_trials"],
            "gate_objective":     GATE_OBJECTIVE,
            "dir_objective":      DIR_OBJECTIVE,
            "gate_models":        ALL_GATE_MODELS,
            "reg_models":         ALL_REG_MODELS,
            "models_gate_run":    MODELS_TO_TUNE_GATE or ALL_GATE_MODELS,
            "models_reg_run":     MODELS_TO_TUNE_REG  or ALL_REG_MODELS,
            "training_target":    TRAIN_TARGET,
            "note": (
                "v8: gate(AUC) + reg_all(MAE, все бары) + "
                "reg_jump(MAE, только скачки + price_ctx)"
            ),
            "cv": {
                "type":       "purged_walk_forward_embargo",
                "n_splits":   N_WF_SPLITS,
                "embargo_1h": EMBARGO_1H,
                "embargo_6h": EMBARGO_6H,
            },
            "total_minutes":   round(total_min, 1),
            "previous_created": old_meta.get("created"),
        },
        "results": merged,
    }

    with open(OUTPUT_FILE, "w") as f:
        json.dump(payload, f, indent=2, default=str)

    print(f"\n{'=' * 78}")
    print(f"  ИТОГ ТЮНИНГА — {total_min:.1f} мин")
    print(f"{'=' * 78}")

    print(
        f"\n  {'TF':<4}{'STAGE':<10}{'модель':<13}"
        f"{'BEST':>10}{'trials':>8}"
    )
    print("  " + "─" * 50)
    for tf, by_t in result.items():
        for tcol, by_s in by_t.items():
            for stage in ["gate", "dir", "dir_jump"]:
                by_m = by_s.get(stage, {})
                for mname, r in by_m.items():
                    am = r["all_metrics_at_best"]
                    if stage == "gate":
                        score_str = (
                            f"AUC="
                            f"{am.get('auc', float('nan')):.4f}"
                        )
                    else:
                        score_str = (
                            f"MAE="
                            f"{am.get('mae', float('nan')):.5f} "
                            f"signA="
                            f"{am.get('sign_acc', float('nan')):.3f}"
                        )
                    print(
                        f"  {tf.upper():<4}{stage:<10}{mname:<13}"
                        f"  {score_str:<28}"
                        f"{r['n_trials']:>8}"
                    )

    print(f"\n{'=' * 78}")
    print(f"Сохранено: {OUTPUT_FILE}")
    print(
        "  Структура JSON:\n"
        "    results[tf][t_static]['gate'][model]     → gate params\n"
        "    results[tf][t_static]['dir'][model]      → reg_all params\n"
        "    results[tf][t_static]['dir_jump'][model] → reg_jump params"
    )
    print(
        "  training_block_v8.py подхватит параметры автоматически."
    )
    print(f"{'=' * 78}")


########################################################################
#  ТЮНИНГ v8 — gate + reg_all + reg_jump
#  Бюджет: 'light' | trials/study: 40 | timeout: 8 мин
#  Gate-модели:  ['xgb', 'lgbm', 'cat']
#  Reg-модели:   ['xgb_huber', 'lgbm_huber', 'hgb_mae', 'elasticnet']
#  Studies всего: 22 (2 ТФ × 11)
#  Старт: 2026-05-23 02:50:10
########################################################################
  [1H-all] вход: final_dataset_1h.csv
  [6H-all] вход: final_dataset_6h.csv
  [1H-jump] вход: final_dataset_1h_jump.csv
  [6H-jump] вход: final_dataset_6h_jump.csv

  ТАЙМФРЕЙМ 1H — БЮДЖЕТ 'light' | embargo=1
  [ALL]  строк=12245 | признаков=45
  fwd_ret: mean=-0.00000 std=0.00480 q01=-0.0145 q99=0.0137
  t_static: {-1: 481, 0: 11318, 1: 446}
  [JUMP] строк=927 | признаков=54 (price_ctx=1)
  [JUMP] t_static: {-1: 481, 1: 446}

  ── GATE (AUC, maximize) ──
    xgb     : AUC=0.7758 | trials=40 | 133s
    lgbm    : AUC=0.7739 | trials=40 | 66s
    cat     : AUC=0.7610 | trials=40 

In [4]:
"""

БЛОК ОБУЧЕНИЯ — КАСКАД С ПАРАЛЛЕЛЬНЫМИ ДАТАСЕТАМИ

Архитектура:

  ДАТАСЕТЫ (4 файла из Блока 3):
    final_dataset_1h.csv      — gate + reg_all для 1H (все бары)
    final_dataset_6h.csv      — gate + reg_all для 6H (все бары)
    final_dataset_1h_jump.csv — reg_jump для 1H (только скачки + цена)
    final_dataset_6h_jump.csv — reg_jump для 6H (только скачки + цена)

  МОДЕЛИ (на каждый ТФ):

    GATE:     бинарная классификация y = (t_static != 0).
              Кандидаты: XGB / LightGBM / CatBoost.
              Выбор по WF-OOS AUC.

    REG-ALL:  регрессия по fwd_ret на ВСЕХ барах (Huber loss).
              Кандидаты: xgb_huber / lgbm_huber / hgb_mae / elasticnet.
              Выбор по WF-OOS MAE.

    REG-JUMP: регрессия по fwd_ret ТОЛЬКО на скачках + ценовой контекст.
              Те же кандидаты, что и REG-ALL.
              Выбор по WF-OOS MAE.
              Обучается на final_dataset_*h_jump.csv.

  ИНФЕРЕНС (два регрессора, два режима tau):

    Каскад-A: GATE → REG-ALL  → tau_all_static  / tau_all_dynamic
    Каскад-B: GATE → REG-JUMP → tau_jump_static / tau_jump_dynamic

    Финальный сигнал:
      casc = sign(reg_pred) если |reg_pred| >= tau И gate == 1

  ОЦЕНКА (static / dynamic):
    static:  tau подобран по t_static (фиксированный порог)
    dynamic: tau подобран по t_dynamic (порог = K * vol(t-1))
             gate + reg НЕ переобучаются — разница только в tau

  БАНДЛ (cascade_bundle_v8.pkl):
    На каждый ТФ:
      g_deploy       — финальная gate-модель (на всей истории)
      r_deploy_all   — финальная REG-ALL модель
      r_deploy_jump  — финальная REG-JUMP модель
      taus_all       — {'static': ..., 'dynamic': ...}
      taus_jump      — {'static': ..., 'dynamic': ...}
      feat_cols      — признаки для gate + reg_all
      feat_cols_jump — признаки для reg_jump (включают price_*)
      + метаданные инференса

ПОЧЕМУ ДВА РЕГРЕССОРА:
  REG-ALL видит все бары, включая флэтовые — он «размыт» нулями.
  REG-JUMP обучен только на реальных движениях и знает ценовой
  контекст (price_log, price_rank, price_vs_sma_ratio, price_zscore).
=============================================================================
"""

import json
import os
import pickle
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform
from xgboost import XGBClassifier, XGBRegressor

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    _HAS_LGBM = True
except ImportError:
    _HAS_LGBM = False

warnings.filterwarnings("ignore")


# КОНФИГ

DATA_1H      = "final_dataset_1h.csv"
DATA_6H      = "final_dataset_6h.csv"
DATA_1H_JUMP = "final_dataset_1h_jump.csv"
DATA_6H_JUMP = "final_dataset_6h_jump.csv"

# Параметры тюнинга
LOAD_TUNED_PARAMS      = True
TUNED_PARAMS_FILE_GATE = "best_params.json"
TUNED_PARAMS_FILE_REG  = "best_params_reg.json"

# Колонки
FWD_COL    = "fwd_ret"
TRAIN_TARGET = "t_static"
EVAL_MODES = ["static", "dynamic"]

# Параметры разметки таргетов
STATIC_THRESH_1H  = 0.008
STATIC_THRESH_6H  = 0.015
DYNAMIC_K_1H      = 1.0
DYNAMIC_K_6H      = 1.0
VOL_INDICATOR_1H  = "volatility_1d"
VOL_INDICATOR_6H  = "volatility_7d"
VOL_FALLBACK      = "atr_pct"

# Кросс-валидация
TEST_RATIO   = 0.2
N_WF_SPLITS  = 5
PURGE_1H     = 1
PURGE_6H     = 6
RANDOM_STATE = 42

# Тюнинг
N_ITER_SEARCH = 30
TUNE_MODELS   = True

GATE_SELECTION_METRIC = "auc"
DIR_SELECTION_METRIC  = "mae"

# TAU
TAU_GRID_SIZE    = 41
TAU_MAX_QUANTILE = 0.99

# Huber delta
HUBER_DELTA_1H = 0.006
HUBER_DELTA_6H = 0.015

# Колонки, которые НИКОГДА не попадают в X (просачивание таргета)
_LEAK_PREFIX = ("fwd_ret", "target", "t_static", "t_dynamic")
# Мета-колонки, которые не являются признаками
_NON_FEATURE = {"c_close"}

# Ценовые признаки, добавленные в jump-датасет
_PRICE_CONTEXT_COLS = {
    "price_log", "price_rank",
    "price_vs_sma_ratio", "price_zscore",
}


# 0. ЗАГРУЗКА ОТТЮНЕННЫХ ПАРАМЕТРОВ


def _load_tuned(path: str, label: str) -> dict | None:
    if not LOAD_TUNED_PARAMS or not os.path.exists(path):
        if LOAD_TUNED_PARAMS:
            print(f"  ⚠ {label}: '{path}' не найден — fallback на дефолты")
        return None
    with open(path) as f:
        data = json.load(f)
    meta = data.get("_meta", {})
    print(
        f"  ✓ {label}: загружено из '{path}' "
        f"(бюджет='{meta.get('budget', '?')}', "
        f"trials={meta.get('n_trials_per_combo', '?')})"
    )
    return data.get("results", {})


_TUNED_GATE: dict | None = None
_TUNED_REG:  dict | None = None


# 1. ЗАГРУЗКА И ТАРГЕТЫ


def _resolve_path(path: str, label: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{label}] не найден '{path}'")
    print(f"  [{label}] вход: {path}")
    return path


def _get_vol(df: pd.DataFrame, preferred: str) -> pd.Series:
    if preferred in df.columns:
        return df[preferred]
    if VOL_FALLBACK in df.columns:
        print(f"  ⚠ '{preferred}' нет — fallback '{VOL_FALLBACK}'")
        return df[VOL_FALLBACK]
    raise KeyError(f"Нет '{preferred}' и '{VOL_FALLBACK}'")


def build_targets(
    df: pd.DataFrame,
    vol_indicator: str,
    dynamic_k: float,
    static_thresh: float,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Добавляет t_static и t_dynamic.
    t_static  — для обучения GATE (фиксированный порог).
    t_dynamic — только для подбора tau в режиме 'dynamic'.
    Возвращает (df_aligned, vol_shift).
    """
    if FWD_COL not in df.columns:
        raise KeyError(f"Нет колонки '{FWD_COL}'")
    df = df.dropna(subset=[FWD_COL]).copy()
    fwd = df[FWD_COL]

    df["t_static"] = np.where(
        fwd > static_thresh, 1,
        np.where(fwd < -static_thresh, -1, 0),
    )

    vol_shift = _get_vol(df, vol_indicator).shift(1)
    valid = vol_shift.notna() & (vol_shift > 0)
    df = df.loc[valid].copy()
    vs = vol_shift.loc[valid]
    fw = df[FWD_COL]

    df["t_dynamic"] = np.where(
        fw > dynamic_k * vs, 1,
        np.where(fw < -dynamic_k * vs, -1, 0),
    )
    return df, vs


def split_features(df: pd.DataFrame) -> list[str]:
    """Признаки для gate + reg_all (без leakage и мета-колонок)."""
    return [
        c for c in df.columns
        if c not in _NON_FEATURE
        and not c.startswith(_LEAK_PREFIX)
    ]


def split_features_jump(df: pd.DataFrame) -> list[str]:
    """
    Признаки для reg_jump.
    Включают ценовой контекст (price_*), но не таргеты.
    """
    return [
        c for c in df.columns
        if c not in _NON_FEATURE
        and not c.startswith(_LEAK_PREFIX)
    ]


def make_gate_xy(
    df: pd.DataFrame,
    target_col: str,
    feat_cols: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """Gate: все бары, y=1 если движение (|fwd_ret| > static_thresh)."""
    X = df[feat_cols].copy()
    y = (df[target_col] != 0).astype(int)
    ok = X.notna().all(axis=1)
    return X.loc[ok], y.loc[ok]


def make_reg_all_xy(
    df: pd.DataFrame,
    feat_cols: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """REG-ALL: все бары, y=fwd_ret."""
    X = df[feat_cols].copy()
    y = df[FWD_COL].copy()
    ok = X.notna().all(axis=1) & y.notna()
    return X.loc[ok], y.loc[ok]


def make_reg_jump_xy(
    df_jump: pd.DataFrame,
    feat_cols_jump: list[str],
) -> tuple[pd.DataFrame, pd.Series]:
    """
    REG-JUMP: только скачки (уже отфильтрованы в Блоке 3).
    y=fwd_ret. Признаки включают price_*.
    """
    X = df_jump[feat_cols_jump].copy()
    y = df_jump[FWD_COL].copy()
    ok = X.notna().all(axis=1) & y.notna()
    return X.loc[ok], y.loc[ok]



# 2. PURGED WALK-FORWARD


def purged_wf_indices(
    n: int, n_splits: int, purge: int,
) -> tuple[np.ndarray, np.ndarray]:
    fold = n // (n_splits + 1)
    for k in range(1, n_splits + 1):
        cut    = fold * k
        tr_end = cut - purge
        va_end = min(cut + fold, n)
        if tr_end <= 0 or va_end <= cut:
            continue
        yield np.arange(0, tr_end), np.arange(cut, va_end)


class PurgedWalkForward:
    def __init__(self, n_splits: int, purge: int):
        self.n_splits = n_splits
        self.purge    = purge

    def split(self, X, y=None, groups=None):
        for tr, va in purged_wf_indices(len(X), self.n_splits, self.purge):
            yield tr, va

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits



# 3. КАНДИДАТЫ — GATE


def _spw(y: pd.Series) -> float:
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    return (neg / pos) if pos > 0 else 1.0


def gate_specs(spw: float) -> dict:
    specs = {
        "xgb": {
            "estimator": XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=spw,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ),
            "param_dist": {
                "n_estimators":      randint(150, 500),
                "max_depth":         randint(3, 8),
                "learning_rate":     uniform(0.01, 0.15),
                "subsample":         uniform(0.6, 0.4),
                "colsample_bytree":  uniform(0.6, 0.4),
                "min_child_weight":  randint(1, 10),
            },
        },
        "cat": {
            "estimator": CatBoostClassifier(
                loss_function="Logloss",
                scale_pos_weight=spw,
                random_state=RANDOM_STATE,
                verbose=0,
            ),
            "param_dist": {
                "iterations":    randint(150, 500),
                "depth":         randint(3, 8),
                "learning_rate": uniform(0.01, 0.15),
                "l2_leaf_reg":   uniform(1, 9),
            },
        },
    }
    if _HAS_LGBM:
        specs["lgbm"] = {
            "estimator": LGBMClassifier(
                objective="binary",
                is_unbalance=True,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            ),
            "param_dist": {
                "n_estimators": randint(150, 500),
                "num_leaves":   randint(15, 80),
                "max_depth":    [-1, 4, 6, 8, 12],
                "learning_rate": uniform(0.01, 0.15),
                "subsample":    uniform(0.6, 0.4),
            },
        }
    return specs


# 4. КАНДИДАТЫ — РЕГРЕССОРЫ (REG-ALL и REG-JUMP)

def reg_specs(huber_delta: float) -> dict:
    specs = {
        "xgb_huber": {
            "estimator": XGBRegressor(
                objective="reg:pseudohubererror",
                huber_slope=huber_delta,
                n_estimators=400,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                tree_method="hist",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            "param_dist": {
                "n_estimators":     randint(200, 700),
                "max_depth":        randint(3, 8),
                "learning_rate":    uniform(0.01, 0.10),
                "subsample":        uniform(0.6, 0.4),
                "colsample_bytree": uniform(0.6, 0.4),
                "reg_alpha":        uniform(0.001, 1.0),
                "reg_lambda":       uniform(0.1, 5.0),
            },
        },
        "hgb_mae": {
            "estimator": HistGradientBoostingRegressor(
                loss="absolute_error",
                max_iter=400,
                max_depth=6,
                learning_rate=0.03,
                l2_regularization=1.0,
                random_state=RANDOM_STATE,
            ),
            "param_dist": {
                "max_iter":          randint(200, 700),
                "max_depth":         [3, 5, 6, 8, None],
                "learning_rate":     uniform(0.01, 0.10),
                "min_samples_leaf":  randint(20, 100),
                "l2_regularization": uniform(0.1, 5.0),
            },
        },
        "elasticnet": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", ElasticNet(
                    alpha=0.001,
                    l1_ratio=0.5,
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_dist": {
                "model__alpha":    uniform(0.0001, 0.05),
                "model__l1_ratio": uniform(0.1, 0.8),
            },
        },
    }
    if _HAS_LGBM:
        specs["lgbm_huber"] = {
            "estimator": LGBMRegressor(
                objective="huber",
                alpha=huber_delta,
                n_estimators=400,
                num_leaves=31,
                learning_rate=0.03,
                subsample=0.85,
                subsample_freq=1,
                colsample_bytree=0.85,
                reg_alpha=0.1,
                reg_lambda=1.0,
                min_data_in_leaf=30,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            ),
            "param_dist": {
                "n_estimators":      randint(200, 700),
                "num_leaves":        randint(15, 80),
                "max_depth":         [-1, 4, 6, 8, 12],
                "learning_rate":     uniform(0.01, 0.10),
                "subsample":         uniform(0.6, 0.4),
                "colsample_bytree":  uniform(0.6, 0.4),
                "reg_alpha":         uniform(0.001, 1.0),
                "reg_lambda":        uniform(0.1, 5.0),
                "min_data_in_leaf":  randint(10, 80),
            },
        }
    return specs


def tune_candidate(
    spec: dict,
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    cv: PurgedWalkForward,
    scoring: str,
) -> object:
    if not TUNE_MODELS:
        return clone(spec["estimator"])
    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["param_dist"],
        n_iter=N_ITER_SEARCH,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
        error_score="raise",
    )
    search.fit(X_tr, y_tr)
    return search.best_estimator_


# 5. МЕТРИКИ

def _binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n_cls = len(np.unique(y_true))
    out = {
        "n":         len(y_true),
        "acc":       accuracy_score(y_true, y_pred),
        "f1_macro":  f1_score(y_true, y_pred, average="macro", zero_division=0),
        "prec_pos":  precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "rec_pos":   recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    if n_cls < 2:
        out["mcc"] = float("nan")
        out["auc"] = float("nan")
    else:
        out["mcc"] = matthews_corrcoef(y_true, y_pred)
        try:
            out["auc"] = roc_auc_score(y_true, y_proba)
        except ValueError:
            out["auc"] = float("nan")
    return out


def _reg_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask = np.abs(y_true) > 1e-9
    sign_acc = (
        float(np.mean(np.sign(y_pred[mask]) == np.sign(y_true[mask])))
        if mask.sum() > 0 else float("nan")
    )
    return {"n": len(y_true), "mae": mae, "rmse": rmse, "sign_acc": sign_acc}


# 6. WF-OOS ПРОГНОЗЫ


def wf_oos_clf(
    estimator,
    X: pd.DataFrame,
    y: pd.Series,
    cv: PurgedWalkForward,
) -> tuple:
    yt, yp, ypr, idx = [], [], [], []
    for tr, va in purged_wf_indices(len(X), cv.n_splits, cv.purge):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        Xva, yva = X.iloc[va], y.iloc[va]
        if ytr.nunique() < 2 or len(Xva) == 0:
            continue
        m = clone(estimator)
        m.fit(Xtr, ytr)
        pr = m.predict_proba(Xva)[:, 1]
        yt.append(yva.to_numpy())
        yp.append((pr >= 0.5).astype(int))
        ypr.append(pr)
        idx.append(va)
    if not yt:
        return None, None, None, None
    return (
        np.concatenate(yt),
        np.concatenate(yp),
        np.concatenate(ypr),
        np.concatenate(idx),
    )


def wf_oos_reg(
    estimator,
    X: pd.DataFrame,
    y: pd.Series,
    cv: PurgedWalkForward,
) -> tuple:
    yt, yp, idx = [], [], []
    for tr, va in purged_wf_indices(len(X), cv.n_splits, cv.purge):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        Xva, yva = X.iloc[va], y.iloc[va]
        if len(Xva) == 0:
            continue
        m = clone(estimator)
        m.fit(Xtr, ytr)
        pr = m.predict(Xva)
        yt.append(yva.to_numpy())
        yp.append(pr)
        idx.append(va)
    if not yt:
        return None, None, None
    return (
        np.concatenate(yt),
        np.concatenate(yp),
        np.concatenate(idx),
    )


# 7. АВТОВЫБОР — GATE


def select_gate(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    cv: PurgedWalkForward,
    tf_key: str,
) -> tuple:
    spw   = _spw(y_tr)
    specs = gate_specs(spw)
    loaded = None

    if (
        _TUNED_GATE
        and tf_key in _TUNED_GATE
        and TRAIN_TARGET in _TUNED_GATE[tf_key]
        and "gate" in _TUNED_GATE[tf_key][TRAIN_TARGET]
    ):
        loaded = _TUNED_GATE[tf_key][TRAIN_TARGET]["gate"]
        print(
            f"\n  [GATE] spw={spw:.3f} | параметры из JSON "
            f"({TRAIN_TARGET}): {list(loaded)}"
        )
    else:
        print(
            f"\n  [GATE] spw={spw:.3f} | тюнинг на месте "
            f"(нет параметров для {tf_key}/{TRAIN_TARGET}/gate)"
        )

    table, tuned = [], {}
    for name, spec in specs.items():
        if loaded and name in loaded:
            params = dict(loaded[name].get("best_params", {}))
            est = clone(spec["estimator"]).set_params(**params)
        else:
            est = tune_candidate(spec, X_tr, y_tr, cv, scoring="f1")
        tuned[name] = est

        yt, yp, ypr, _ = wf_oos_clf(est, X_tr, y_tr, cv)
        if yt is None:
            print(f"    {name:5}: WF-OOS пуст")
            continue
        m = _binary_metrics(yt, yp, ypr)
        m["model"] = name
        table.append(m)
        print(
            f"    {name:5}: MCC={m['mcc']:.4f} "
            f"F1m={m['f1_macro']:.4f} AUC={m['auc']:.4f} "
            f"P+={m['prec_pos']:.3f} R+={m['rec_pos']:.3f}"
        )

    if not table:
        raise RuntimeError("GATE: ни один кандидат не оценён")

    def _key(r):
        v = r.get(GATE_SELECTION_METRIC, float("nan"))
        return -1e9 if v != v else v

    best  = max(table, key=_key)
    bname = best["model"]
    print(
        f"  [GATE] ЛУЧШАЯ по {GATE_SELECTION_METRIC.upper()}: "
        f"{bname} ({GATE_SELECTION_METRIC}={best[GATE_SELECTION_METRIC]:.4f})"
    )
    return bname, tuned[bname], pd.DataFrame(table)


# 8. АВТОВЫБОР — РЕГРЕССОР (единая функция для REG-ALL и REG-JUMP)

def select_reg(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    cv: PurgedWalkForward,
    tf_key: str,
    huber_delta: float,
    reg_label: str = "REG-ALL",
    params_key: str = "dir",
) -> tuple:
    """
    Автовыбор регрессора.

    reg_label: строка для вывода ("REG-ALL" или "REG-JUMP")
    params_key: ключ в JSON ("dir" для reg_all, "dir_jump" для reg_jump)
    """
    specs  = reg_specs(huber_delta)
    loaded = None

    if (
        _TUNED_REG
        and tf_key in _TUNED_REG
        and TRAIN_TARGET in _TUNED_REG[tf_key]
        and params_key in _TUNED_REG[tf_key][TRAIN_TARGET]
    ):
        loaded = _TUNED_REG[tf_key][TRAIN_TARGET][params_key]
        print(
            f"\n  [{reg_label}] параметры из JSON "
            f"({TRAIN_TARGET}/{params_key}): {list(loaded)}"
        )
    else:
        print(
            f"\n  [{reg_label}] тюнинг на месте (neg_MAE) "
            f"(нет параметров для {tf_key}/{TRAIN_TARGET}/{params_key})"
        )

    table, tuned, oos_preds = [], {}, {}
    for name, spec in specs.items():
        if loaded and name in loaded:
            params = dict(loaded[name].get("best_params", {}))
            est = clone(spec["estimator"]).set_params(**params)
        else:
            est = tune_candidate(
                spec, X_tr, y_tr, cv,
                scoring="neg_mean_absolute_error",
            )
        tuned[name] = est

        yt, yp, idx = wf_oos_reg(est, X_tr, y_tr, cv)
        if yt is None:
            print(f"    {name:11}: WF-OOS пуст")
            continue
        m = _reg_metrics(yt, yp)
        m["model"] = name
        table.append(m)
        oos_preds[name] = pd.Series(
            yp, index=X_tr.index[idx], name=name
        )
        print(
            f"    {name:11}: MAE={m['mae']:.5f} "
            f"RMSE={m['rmse']:.5f} sign_acc={m['sign_acc']:.3f}"
        )

    if not table:
        raise RuntimeError(f"{reg_label}: ни один кандидат не оценён")

    if DIR_SELECTION_METRIC in ("mae", "rmse"):
        best = min(table, key=lambda r: r[DIR_SELECTION_METRIC])
    else:
        best = max(table, key=lambda r: r[DIR_SELECTION_METRIC])
    bname = best["model"]
    print(
        f"  [{reg_label}] ЛУЧШАЯ по {DIR_SELECTION_METRIC.upper()}: "
        f"{bname} ({DIR_SELECTION_METRIC}={best[DIR_SELECTION_METRIC]:.5f})"
    )
    return bname, tuned[bname], pd.DataFrame(table), oos_preds[bname]


# 9. ПОДБОР TAU

def tune_tau(
    gate_oos: pd.Series,
    reg_oos: pd.Series,
    y3_train: pd.Series,
    idx_common: pd.Index,
    label: str = "",
) -> tuple:
    """
    Подбирает tau на |reg_pred|, максимизируя MCC каскада на 3 классах.
    gate_oos / reg_oos — WF-OOS прогнозы на train-части.
    y3_train — серия 3-классов (-1/0/+1).
    """
    g = gate_oos.loc[idx_common].to_numpy()
    r = reg_oos.loc[idx_common].to_numpy()
    y = y3_train.loc[idx_common].to_numpy()

    tau_max  = float(np.quantile(np.abs(r), TAU_MAX_QUANTILE))
    tau_grid = np.linspace(0.0, tau_max, TAU_GRID_SIZE)

    rows = []
    for tau in tau_grid:
        casc = np.zeros_like(y, dtype=int)
        mv   = (g == 1) & (np.abs(r) >= tau)
        casc[mv] = np.where(r[mv] > 0, 1, -1)
        mcc = (
            matthews_corrcoef(y, casc)
            if len(np.unique(y)) > 1 else float("nan")
        )
        f1m    = f1_score(y, casc, labels=[-1, 0, 1],
                          average="macro", zero_division=0)
        active = float(np.mean(casc != 0))
        rows.append({
            "tau": tau, "mcc": mcc,
            "f1_macro": f1m, "active_rate": active,
        })

    grid     = pd.DataFrame(rows)
    best_row = grid.loc[grid["mcc"].idxmax()]
    print(
        f"  [TAU/{label}] best_tau={best_row['tau']:.6f} "
        f"→ MCC={best_row['mcc']:.4f} "
        f"F1m={best_row['f1_macro']:.4f} "
        f"active={best_row['active_rate']:.3f}"
    )
    return float(best_row["tau"]), grid, float(best_row["mcc"])


# 10. ОТЧЁТЫ

def report(
    title: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: list,
    names: list,
) -> None:
    print(f"\n  ── {title} ──")
    print(classification_report(
        y_true, y_pred,
        labels=labels, target_names=names,
        digits=3, zero_division=0,
    ))
    print("  Confusion (строки=факт, столбцы=прогноз):")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(pd.DataFrame(
        cm,
        index=[f"факт {n}" for n in names],
        columns=[f"пр.{n}" for n in names],
    ))


# 11. ОЦЕНКА КАСКАДА НА HOLD-OUT

def eval_cascade(
    gate_pred: np.ndarray,
    reg_pred: np.ndarray,
    y3_te: pd.Series,
    tau: float,
    mode_label: str,
    index: pd.Index,
) -> dict:
    casc = np.zeros(len(y3_te), dtype=int)
    mv   = (gate_pred == 1) & (np.abs(reg_pred) >= tau)
    casc[mv] = np.where(reg_pred[mv] > 0, 1, -1)
    casc_s = pd.Series(casc, index=index)

    acc    = accuracy_score(y3_te, casc)
    f1m    = f1_score(y3_te, casc, labels=[-1, 0, 1],
                      average="macro", zero_division=0)
    mcc    = (
        matthews_corrcoef(y3_te, casc)
        if y3_te.nunique() > 1 else float("nan")
    )
    active = float((casc != 0).mean())

    print(
        f"\n  [КАСКАД-{mode_label} / HOLD-OUT] "
        f"ACC={acc:.3f} F1m={f1m:.3f} MCC={mcc:.4f} | "
        f"active={active:.3f} (tau={tau:.5f})"
    )
    report(
        f"КАСКАД-{mode_label} (-1=DOWN, 0=FLAT, +1=UP)",
        y3_te, casc_s, [-1, 0, 1], ["DOWN", "FLAT", "UP"],
    )
    return {
        "acc": acc, "f1_macro": f1m, "mcc": mcc,
        "active_rate": active, "tau": tau,
    }


# 12. РАН ПО ОДНОМУ ТАЙМФРЕЙМУ


def run_timeframe(
    name: str,
    path_all: str,
    path_jump: str,
    vol_indicator: str,
    dynamic_k: float,
    static_thresh: float,
    purge: int,
    huber_delta: float,
) -> dict:
    """
    Полный цикл обучения для одного ТФ.

    path_all  — final_dataset_*h.csv      (все бары)
    path_jump — final_dataset_*h_jump.csv (только скачки + price_*)
    """
    print("\n" + "=" * 72)
    print(f"  ТАЙМФРЕЙМ {name.upper()} | huber_delta={huber_delta}")
    print(f"  all={path_all} | jump={path_jump}")
    print("=" * 72)

    # Загрузка all-bars датасета
    df_all = pd.read_csv(path_all, index_col=0, parse_dates=True)
    df_all, vol_shift = build_targets(
        df_all, vol_indicator, dynamic_k, static_thresh
    )
    feat_cols = split_features(df_all)

    print(f"  [ALL]  строк={len(df_all)} | признаков={len(feat_cols)}")
    for t in ["t_static", "t_dynamic"]:
        vc  = df_all[t].value_counts().sort_index().to_dict()
        sig = (df_all[t] != 0).mean()
        print(f"  {t:11}: {vc} | signal_rate={sig:.3f}")
    print(
        f"  fwd_ret: mean={df_all[FWD_COL].mean():.5f} "
        f"std={df_all[FWD_COL].std():.5f} "
        f"q01={df_all[FWD_COL].quantile(0.01):.4f} "
        f"q99={df_all[FWD_COL].quantile(0.99):.4f}"
    )

    # Загрузка jump-датасета
    df_jump = pd.read_csv(path_jump, index_col=0, parse_dates=True)
    df_jump, _ = build_targets(
        df_jump, vol_indicator, dynamic_k, static_thresh
    )
    feat_cols_jump = split_features_jump(df_jump)

    # Добавляем ценовой контекст в jump-признаки (если они есть в датасете)
    present_price_ctx = [
        c for c in _PRICE_CONTEXT_COLS if c in df_jump.columns
    ]
    feat_cols_jump = list(dict.fromkeys(
        feat_cols_jump + present_price_ctx
    ))

    print(
        f"  [JUMP] строк={len(df_jump)} | "
        f"признаков={len(feat_cols_jump)} "
        f"(в т.ч. price_ctx={len(present_price_ctx)})"
    )
    vc_jump = df_jump["t_static"].value_counts().sort_index().to_dict()
    print(f"  [JUMP] t_static: {vc_jump}")

    cv = PurgedWalkForward(N_WF_SPLITS, purge)

    # Хронологический split
    n_all  = len(df_all)
    cut    = int(n_all * (1.0 - TEST_RATIO))
    df_tr  = df_all.iloc[:cut]
    df_te  = df_all.iloc[cut:]

    # jump split: строки, чей индекс попадает в train/test по df_all
    train_end = df_tr.index[-1]
    df_jump_tr = df_jump.loc[df_jump.index <= train_end]
    df_jump_te = df_jump.loc[df_jump.index > train_end]

    print(
        f"\n  all:  train={len(df_tr)} test={len(df_te)}"
    )
    print(
        f"  jump: train={len(df_jump_tr)} test={len(df_jump_te)}"
    )
    print(
        f"  train t_static : "
        f"{df_tr['t_static'].value_counts().sort_index().to_dict()}"
    )
    print(
        f"  train t_dynamic: "
        f"{df_tr['t_dynamic'].value_counts().sort_index().to_dict()}"
    )

    # GATE
    print(f"\n  {'─' * 64}")
    print(f"  GATE (train target={TRAIN_TARGET}, все бары)")
    print(f"  {'─' * 64}")

    Xg_tr, yg_tr = make_gate_xy(df_tr, TRAIN_TARGET, feat_cols)
    g_name, g_model, g_tbl = select_gate(Xg_tr, yg_tr, cv, tf_key=name)

    # WF-OOS gate-прогнозы (общие для подбора tau обоих каскадов)
    _, yp_g, _, idx_g = wf_oos_clf(g_model, Xg_tr, yg_tr, cv)
    gate_oos = pd.Series(
        yp_g, index=Xg_tr.index[idx_g], name="gate"
    )

    # REG-ALL
    print(f"\n  {'─' * 64}")
    print(f"  REG-ALL (все бары, y=fwd_ret)")
    print(f"  {'─' * 64}")

    Xr_tr, yr_tr = make_reg_all_xy(df_tr, feat_cols)
    r_name_all, r_model_all, r_tbl_all, r_oos_all = select_reg(
        Xr_tr, yr_tr, cv,
        tf_key=name, huber_delta=huber_delta,
        reg_label="REG-ALL", params_key="dir",
    )

    # REG-JUMP
    print(f"\n  {'─' * 64}")
    print(f"  REG-JUMP (только скачки, y=fwd_ret + price_context)")
    print(f"  {'─' * 64}")

    Xrj_tr, yrj_tr = make_reg_jump_xy(df_jump_tr, feat_cols_jump)
    r_name_jump, r_model_jump, r_tbl_jump, r_oos_jump = select_reg(
        Xrj_tr, yrj_tr, cv,
        tf_key=name, huber_delta=huber_delta,
        reg_label="REG-JUMP", params_key="dir_jump",
    )

    # ПОДБОР TAU (два каскада × два режима)
    print(f"\n  {'─' * 64}")
    print("  ПОДБОР TAU (WF-OOS train)")
    print(f"  {'─' * 64}")

    taus_all:  dict[str, float] = {}
    taus_jump: dict[str, float] = {}
    tau_mccs:  dict[str, dict]  = {}

    for mode in EVAL_MODES:
        y3_tr = df_tr[f"t_{mode}"].astype(int)

        # TAU для каскада-A (gate + reg_all)
        common_all = (
            gate_oos.index
            .intersection(r_oos_all.index)
            .intersection(df_tr.index)
        )
        if len(common_all) < 100:
            raise RuntimeError(
                f"[{name}/{mode}/all] Мало OOS-точек: {len(common_all)}"
            )
        tau_a, grid_a, mcc_a = tune_tau(
            gate_oos, r_oos_all, y3_tr,
            common_all, label=f"{mode}/all",
        )
        taus_all[mode] = tau_a

        common_jump = (
            gate_oos.index
            .intersection(r_oos_jump.index)
            .intersection(df_tr.index)
        )
        if len(common_jump) >= 50:
            tau_j, grid_j, mcc_j = tune_tau(
                gate_oos, r_oos_jump, y3_tr,
                common_jump, label=f"{mode}/jump",
            )
            taus_jump[mode] = tau_j
            print(
                f"  [TAU сравнение / {mode}]  "
                f"all:  tau={tau_a:.5f} MCC={mcc_a:.4f} | "
                f"jump: tau={tau_j:.5f} MCC={mcc_j:.4f}"
            )
        else:
            taus_jump[mode] = tau_a
            mcc_j = float("nan")
            print(
                f"  ⚠ [{mode}/jump] Мало OOS-точек "
                f"({len(common_jump)}) → tau_jump = tau_all"
            )

        tau_mccs[mode] = {
            "all":  mcc_a,
            "jump": mcc_j,
        }

    # ФИНАЛЬНЫЕ МОДЕЛИ на train
    g_final    = clone(g_model).fit(Xg_tr, yg_tr)
    r_final_all  = clone(r_model_all).fit(Xr_tr, yr_tr)
    r_final_jump = clone(r_model_jump).fit(Xrj_tr, yrj_tr)

    print("  HOLD-OUT: gate + reg_all + reg_jump × 2 режима tau")

    Xte_all = df_te[feat_cols].copy()
    ok_te   = Xte_all.notna().all(axis=1)
    Xte_all = Xte_all.loc[ok_te]
    yfwd_te = df_te.loc[ok_te, FWD_COL]

    # Gate-предсказания
    gate_pred  = g_final.predict(Xte_all)
    gate_proba = g_final.predict_proba(Xte_all)[:, 1]

    # Оценка gate vs t_static
    gate_true_s = (df_te.loc[ok_te, "t_static"] != 0).astype(int)
    m_gate = _binary_metrics(gate_true_s, gate_pred, gate_proba)
    print(
        f"\n  [GATE / HOLD-OUT vs t_static] модель={g_name} | "
        f"MCC={m_gate['mcc']:.4f} F1m={m_gate['f1_macro']:.4f} "
        f"AUC={m_gate['auc']:.4f} "
        f"P(move)={m_gate['prec_pos']:.3f} "
        f"R(move)={m_gate['rec_pos']:.3f}"
    )

    # REG-ALL предсказания на hold-out
    reg_pred_all = r_final_all.predict(Xte_all)
    m_reg_all    = _reg_metrics(yfwd_te.to_numpy(), reg_pred_all)

    # REG-JUMP предсказания на hold-out

    Xte_jump_raw = df_jump_te[feat_cols_jump].copy()
    ok_te_jump   = Xte_jump_raw.notna().all(axis=1)
    Xte_jump     = Xte_jump_raw.loc[ok_te_jump]

    reg_pred_jump_sparse = r_final_jump.predict(Xte_jump)
    reg_pred_jump_full = reg_pred_all.copy()
    jump_mask_te = Xte_all.index.isin(Xte_jump.index)
    jump_idx_common = Xte_all.index[jump_mask_te]
    jump_idx_in_Xte_jump = Xte_jump.index.intersection(jump_idx_common)
    if len(jump_idx_in_Xte_jump) > 0:
        pos_in_all  = np.where(jump_mask_te)[0]
        pos_in_jump = [
            Xte_jump.index.get_loc(i)
            for i in jump_idx_in_Xte_jump
        ]
        reg_pred_jump_full[pos_in_all] = reg_pred_jump_sparse[pos_in_jump]

    m_reg_jump = _reg_metrics(yfwd_te.to_numpy(), reg_pred_jump_full)

    for reg_label, reg_pred_arr, m_reg in [
        ("ALL",  reg_pred_all,       m_reg_all),
        ("JUMP", reg_pred_jump_full, m_reg_jump),
    ]:
        for mode in EVAL_MODES:
            mv_te = (df_te.loc[ok_te, f"t_{mode}"] != 0)
            if mv_te.sum() > 0:
                m_reg[f"sign_acc_jumps_{mode}"] = float(np.mean(
                    np.sign(reg_pred_arr[mv_te.to_numpy()])
                    == np.sign(yfwd_te[mv_te].to_numpy())
                ))
            else:
                m_reg[f"sign_acc_jumps_{mode}"] = float("nan")

        print(
            f"  [REG-{reg_label} / HOLD-OUT] "
            f"MAE={m_reg['mae']:.5f} RMSE={m_reg['rmse']:.5f} | "
            f"sign_acc(all)={m_reg['sign_acc']:.3f} "
            f"signJ_stat={m_reg.get('sign_acc_jumps_static', float('nan')):.3f} "
            f"signJ_dyn={m_reg.get('sign_acc_jumps_dynamic', float('nan')):.3f}"
        )

    casc_results: dict[str, dict] = {}

    for mode in EVAL_MODES:
        y3_te = df_te.loc[ok_te, f"t_{mode}"].astype(int)

        key_a = f"all_{mode}"
        casc_results[key_a] = eval_cascade(
            gate_pred, reg_pred_all, y3_te,
            taus_all[mode],
            mode_label=f"A-{mode}(all)",
            index=Xte_all.index,
        )

        # Каскад-B: gate + reg_jump
        key_b = f"jump_{mode}"
        casc_results[key_b] = eval_cascade(
            gate_pred, reg_pred_jump_full, y3_te,
            taus_jump[mode],
            mode_label=f"B-{mode}(jump)",
            index=Xte_all.index,
        )

    # DEPLOY-МОДЕЛИ на всей истории
    print(f"\n  {'─' * 64}")
    print("  DEPLOY: переобучение на всей истории")
    print(f"  {'─' * 64}")

    Xg_all, yg_all     = make_gate_xy(df_all, TRAIN_TARGET, feat_cols)
    Xr_all, yr_all     = make_reg_all_xy(df_all, feat_cols)
    Xrj_all, yrj_all   = make_reg_jump_xy(df_jump, feat_cols_jump)

    g_deploy    = clone(g_model).fit(Xg_all, yg_all)
    r_deploy_all  = clone(r_model_all).fit(Xr_all, yr_all)
    r_deploy_jump = clone(r_model_jump).fit(Xrj_all, yrj_all)

    print(
        f"  g_deploy:      {g_name} "
        f"(обучен на {len(Xg_all)} барах)"
    )
    print(
        f"  r_deploy_all:  {r_name_all} "
        f"(обучен на {len(Xr_all)} барах)"
    )
    print(
        f"  r_deploy_jump: {r_name_jump} "
        f"(обучен на {len(Xrj_all)} скачках)"
    )

    return {
        "name":            name,
        "train_target":    TRAIN_TARGET,
        # модели для инференса
        "g_deploy":        g_deploy,
        "r_deploy_all":    r_deploy_all,
        "r_deploy_jump":   r_deploy_jump,
        # промежуточные (для анализа)
        "g_final":         g_final,
        "r_final_all":     r_final_all,
        "r_final_jump":    r_final_jump,
        # имена лучших моделей
        "gate_model_name": g_name,
        "dir_model_name_all":  r_name_all,
        "dir_model_name_jump": r_name_jump,
        # таблицы отбора
        "gate_select_table": g_tbl,
        "reg_all_select_table":  r_tbl_all,
        "reg_jump_select_table": r_tbl_jump,
        # tau для двух каскадов × двух режимов
        "taus_all":   taus_all,
        "taus_jump":  taus_jump,
        "tau_mccs":   tau_mccs,
        # метрики hold-out
        "m_gate":     m_gate,
        "m_reg_all":  m_reg_all,
        "m_reg_jump": m_reg_jump,
        "cascade":    casc_results,
        # признаки
        "feat_cols":      feat_cols,
        "feat_cols_jump": feat_cols_jump,
        # инференс-контекст
        "X_full":         df_all[feat_cols].copy(),
        "X_jump_full":    df_jump[feat_cols_jump].copy(),
        "price_full":     (
            df_all["c_close"].copy()
            if "c_close" in df_all.columns else None
        ),
        "vol_indicator":  (
            vol_indicator if vol_indicator in df_all.columns
            else VOL_FALLBACK
        ),
        "vol_shift_full": vol_shift,
        "dynamic_k":      dynamic_k,
        "static_thresh":  static_thresh,
        "last_ts":        df_all.index[-1],
    }


# 13. ЗАПУСК

if __name__ == "__main__":
    print("\n" + "#" * 72)
    print("#  ОБУЧЕНИЕ v8 — GATE + REG-ALL + REG-JUMP | 4 датасета")
    print("#" * 72)

    _TUNED_GATE = _load_tuned(TUNED_PARAMS_FILE_GATE, "GATE params")
    _TUNED_REG  = _load_tuned(TUNED_PARAMS_FILE_REG,  "REG params")

    _resolve_path(DATA_1H,      "1H-all")
    _resolve_path(DATA_6H,      "6H-all")
    _resolve_path(DATA_1H_JUMP, "1H-jump")
    _resolve_path(DATA_6H_JUMP, "6H-jump")

    RESULTS = {
        "1h": run_timeframe(
            name="1h",
            path_all=DATA_1H,
            path_jump=DATA_1H_JUMP,
            vol_indicator=VOL_INDICATOR_1H,
            dynamic_k=DYNAMIC_K_1H,
            static_thresh=STATIC_THRESH_1H,
            purge=PURGE_1H,
            huber_delta=HUBER_DELTA_1H,
        ),
        "6h": run_timeframe(
            name="6h",
            path_all=DATA_6H,
            path_jump=DATA_6H_JUMP,
            vol_indicator=VOL_INDICATOR_6H,
            dynamic_k=DYNAMIC_K_6H,
            static_thresh=STATIC_THRESH_6H,
            purge=PURGE_6H,
            huber_delta=HUBER_DELTA_6H,
        ),
    }

    # СВОДКА
    print("\n" + "=" * 110)
    print("  ИТОГОВАЯ СВОДКА — 2 ТФ × 2 каскада × 2 режима")
    print("=" * 110)

    # Заголовок
    print(
        f"  {'TF':<4}{'gate':<7}{'reg_all':<13}{'reg_jump':<13}"
        f"{'каскад':<8}{'mode':<9}"
        f"{'tau':>9}{'MCC':>10}{'F1':>9}{'active':>9}"
        f"{'signJ':>8}"
    )
    print("  " + "─" * 100)

    for tf, res in RESULTS.items():
        for casc_type in ["all", "jump"]:
            for mode in EVAL_MODES:
                key = f"{casc_type}_{mode}"
                c   = res["cascade"][key]
                reg_name = (
                    res["dir_model_name_all"]
                    if casc_type == "all"
                    else res["dir_model_name_jump"]
                )
                sj_key = f"sign_acc_jumps_{mode}"
                m_reg  = (
                    res["m_reg_all"]
                    if casc_type == "all"
                    else res["m_reg_jump"]
                )
                sj = m_reg.get(sj_key, float("nan"))
                print(
                    f"  {tf.upper():<4}"
                    f"{res['gate_model_name']:<7}"
                    f"{res['dir_model_name_all']:<13}"
                    f"{res['dir_model_name_jump']:<13}"
                    f"{casc_type:<8}"
                    f"{mode:<9}"
                    f"{c['tau']:>9.5f}"
                    f"{c['mcc']:>10.4f}"
                    f"{c['f1_macro']:>9.3f}"
                    f"{c['active_rate']:>9.3f}"
                    f"{sj:>8.3f}"
                )

    print("=" * 110)
    print(
        "  Каскад-A (all):  GATE → REG-ALL  → tau_all\n"
        "  Каскад-B (jump): GATE → REG-JUMP → tau_jump\n"
        "  REG-JUMP обучен только на барах со скачком + ценовой контекст.\n"
        "  Выбор каскада для деплоя: по MCC на hold-out."
    )
    print("=" * 110)

    # БАНДЛ
    BUNDLE_FILE = "cascade_bundle_v8.pkl"
    bundle = {}

    for tf, res in RESULTS.items():
        bundle[tf] = {
            # идентификация
            "name":         res["name"],
            "train_target": res["train_target"],
            # имена лучших моделей
            "gate_model_name":      res["gate_model_name"],
            "dir_model_name_all":   res["dir_model_name_all"],
            "dir_model_name_jump":  res["dir_model_name_jump"],
            # deploy-модели
            "g_deploy":       res["g_deploy"],
            "r_deploy_all":   res["r_deploy_all"],
            "r_deploy_jump":  res["r_deploy_jump"],
            # tau: 2 каскада × 2 режима
            "taus_all":   res["taus_all"],
            "taus_jump":  res["taus_jump"],
            "tau_mccs":   res["tau_mccs"],
            # метрики
            "m_gate":     res["m_gate"],
            "m_reg_all":  res["m_reg_all"],
            "m_reg_jump": res["m_reg_jump"],
            "cascade":    res["cascade"],
            # признаки
            "feat_cols":      res["feat_cols"],
            "feat_cols_jump": res["feat_cols_jump"],
            # контекст инференса
            "X_full":         res["X_full"],
            "X_jump_full":    res["X_jump_full"],
            "price_full":     res["price_full"],
            "vol_indicator":  res["vol_indicator"],
            "vol_shift_full": res["vol_shift_full"],
            "dynamic_k":      res["dynamic_k"],
            "static_thresh":  res["static_thresh"],
            "last_ts":        res["last_ts"],
        }

    with open(BUNDLE_FILE, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

    sz = os.path.getsize(BUNDLE_FILE) / 1024
    print(f"\n  Бандл → {BUNDLE_FILE} ({sz:.0f} КБ)")
    print(f"     ТФ: {list(bundle)}")
    print(f"     Каскады: all, jump | Режимы tau: {EVAL_MODES}")
    print()
    print("  Инференс (псевдокод):")
    print("    gate_prob = g_deploy.predict_proba(X[feat_cols])[:,1]")
    print("    gate_pred = (gate_prob >= 0.5).astype(int)")
    print()
    print("    # Каскад-A (все бары):")
    print("    reg_a = r_deploy_all.predict(X[feat_cols])")
    print("    casc_a = sign(reg_a) if |reg_a| >= taus_all['static'] and gate==1")
    print()
    print("    # Каскад-B (jump-специализированный):")
    print("    reg_b = r_deploy_jump.predict(X[feat_cols_jump])")
    print("    casc_b = sign(reg_b) if |reg_b| >= taus_jump['static'] and gate==1")

ModuleNotFoundError: No module named 'catboost'

In [ ]:
"""
БЛОК ИНФЕРЕНСА  — прогноз каскадом (заменяет ячейки 3 и 4)

Совместим с бандлом из training_block_v6.py (cascade_bundle.pkl).

Как работает каскадный прогноз на каждый будущий бар:
    1.  ЭТАП 1 (gate)      : P(движение) = g_deploy.predict_proba
    2.  если P(движение) ≥ gate_threshold → ЭТАП 2 (direction):
        P(UP) = d_deploy.predict_proba; знак = UP/DOWN
        иначе → FLAT
    3.  итог: -1 (DOWN) / 0 (FLAT) / +1 (UP) + уверенности обоих этапов

Один блок обслуживает И 1H, И 6H (раньше было два дубля по ~200 строк).

=============================================================================
"""

import os
import pickle
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# конфигурация по таймфреймам
BUNDLE_FILE = "cascade_bundle.pkl"

TF_CONFIG = {
    "1h": {"future_bars": 24, "unit": "ч",  "unit_div": 1.0,
           "lookback": 100},
    "6h": {"future_bars": 28, "unit": "дн", "unit_div": 24.0,
           "lookback": 100},
}

INFER_TARGET = "t_dynamic"

# Порог этапа 1: при P(движение) ниже — считаем FLAT.
GATE_THRESHOLD = 0.50
# Порог этапа 2: уверенность направления для непустого сигнала.
DIR_THRESHOLD = 0.50

PLOT_COLORS = {
    "history": "#ffffff", "anchor": "#9e9e9e",
    "up": "#00e676", "down": "#ff5252", "now": "#ffd600",
}



# 1. ЗАГРУЗКА БАНДЛА


def load_bundle(tf):
    """Берёт RESULTS из памяти, иначе cascade_bundle.pkl."""
    if "RESULTS" in globals():
        src = globals()["RESULTS"]
        print(f"[{tf.upper()}] источник: RESULTS (память)")
        return src[tf]
    if os.path.exists(BUNDLE_FILE):
        with open(BUNDLE_FILE, "rb") as f:
            b = pickle.load(f)
        print(f"[{tf.upper()}] источник: {BUNDLE_FILE}")
        return b[tf]
    raise FileNotFoundError(
        f"Нет RESULTS в памяти и нет '{BUNDLE_FILE}'. "
        f"Сначала выполните training_block_v6."
    )


def get_stage_models(tf_bundle, target):
    """Достаёт deploy-модели каскада для заданного таргета."""
    bt = tf_bundle["by_target"]
    if target not in bt:
        avail = list(bt)
        raise KeyError(
            f"Таргет '{target}' нет в бандле. Доступно: {avail}. "
            f"Измените INFER_TARGET."
        )
    d = bt[target]
    return d["g_deploy"], d["d_deploy"], d


# 2. ПОСТРОЕНИЕ БУДУЩИХ БАРОВ (frozen-features, с честным дисклеймером)

def _find_col(columns, patterns):
    for pat in patterns:
        for col in columns:
            if re.search(pat, col, re.I):
                return col
    return None


def infer_bar_step(index):
    deltas = pd.Series(index[-50:]).diff().dropna()
    if deltas.empty:
        raise ValueError("Недостаточно точек для оценки шага бара.")
    return pd.Timedelta(deltas.median())


def build_future_frame(x_full, n_bars, bar_step):
    """
    Реплицирует последнюю строку фич на n_bars вперёд и обновляет
    ТОЛЬКО календарные/лаговые фичи. Остальные заморожены (см.
    дисклеймер в шапке файла).
    """
    last_ts = x_full.index[-1]
    fut_idx = pd.date_range(last_ts + bar_step, periods=n_bars,
                            freq=bar_step)
    fut = pd.concat([x_full.iloc[[-1]]] * n_bars, ignore_index=True)
    fut.index = fut_idx
    cols = list(fut.columns)
    updated, skipped = [], []

    fwd = _find_col(cols, [r"^fwd_ret"])
    if fwd:
        fut[fwd] = 0.0
        updated.append(fwd)

    weekend = _find_col(cols, [r"weekend"])
    if weekend:
        fut[weekend] = (fut_idx.dayofweek >= 5).astype(int)
        updated.append(weekend)
    else:
        skipped.append("is_weekend")

    sessions = {
        r"session_asia": (0, 8), r"session_europe": (7, 16),
        r"session_ny": (13, 22), r"session_overlap": (13, 16),
    }
    for pat, (lo, hi) in sessions.items():
        col = _find_col(cols, [pat])
        if col:
            h = fut_idx.hour
            fut[col] = ((h >= lo) & (h < hi)).astype(int)
            updated.append(col)
        else:
            skipped.append(pat)

    bs = _find_col(cols, [r"bars_since"])
    if bs:
        base = int(x_full[bs].iloc[-1])
        fut[bs] = np.arange(base + 1, base + 1 + n_bars)
        updated.append(bs)
    else:
        skipped.append("bars_since")

    lags = [c for c in cols if re.search(r"ret_lag", c, re.I)]
    if lags:
        fut[lags] = 0.0
        updated.extend(lags)

    return fut, updated, skipped


# 3. КАСКАДНЫЙ ПРОГНОЗ


def predict_future(tf, tf_bundle, n_bars, unit, unit_div):
    g_model, d_model, meta = get_stage_models(tf_bundle, INFER_TARGET)
    x_full = tf_bundle["X_full"]
    feat_cols = tf_bundle["feature_cols"]
    price_full = tf_bundle["price_full"]

    bar_step = infer_bar_step(x_full.index)
    bar_hours = bar_step.total_seconds() / 3600.0
    fut, updated, skipped = build_future_frame(
        x_full[feat_cols], n_bars, bar_step)

    print(f"[{tf.upper()}] шаг≈{bar_hours:.1f}ч | горизонт {n_bars} "
          f"баров ≈ {n_bars * bar_hours / unit_div:.1f} {unit}")
    print(f"  модели каскада: gate={meta['gate_model']} "
          f"dir={meta['dir_model']} | таргет={INFER_TARGET}")
    print(f"  обновлено фич: {len(set(updated))} | "
          f"заморожено: {len(feat_cols) - len(set(updated))} "
          f"(см. дисклеймер — прогноз вырождается за горизонт)")
    if skipped:
        print(f"  не найдены: {sorted(set(skipped))}")

    # ЭТАП 1: вероятность движения
    p_move = g_model.predict_proba(fut)[:, 1]
    gate_on = p_move >= GATE_THRESHOLD

    # ЭТАП 2: направление (только там, где gate сработал)
    p_up = np.full(n_bars, np.nan)
    if gate_on.any():
        p_up[gate_on] = d_model.predict_proba(fut.loc[gate_on])[:, 1]

    signal = np.zeros(n_bars, dtype=int)
    up_mask = gate_on & (p_up >= DIR_THRESHOLD)
    dn_mask = gate_on & (p_up < DIR_THRESHOLD)
    signal[up_mask] = 1
    signal[dn_mask] = -1

    bar_num = np.arange(1, n_bars + 1)
    res = pd.DataFrame(index=fut.index)
    res["bar_num"] = bar_num
    res["units_ahead"] = bar_num * bar_hours / unit_div
    res["last_known_price"] = (
        float(price_full.iloc[-1]) if price_full is not None else np.nan
    )
    res["p_move"] = p_move
    res["p_up"] = p_up
    res["p_down"] = 1.0 - p_up
    res["signal"] = signal
    res["signal_label"] = res["signal"].map(
        {1: "UP", -1: "DOWN", 0: "FLAT"})
    # уверенность = P(move) * P(направление)
    conf = np.zeros(n_bars)
    conf[up_mask] = p_move[up_mask] * p_up[up_mask]
    conf[dn_mask] = p_move[dn_mask] * (1.0 - p_up[dn_mask])
    res["confidence_pct"] = conf * 100.0
    return res, meta


# 4. ТЕКСТОВАЯ СВОДКА


def print_summary(res, tf, unit, meta):
    total = float(res["units_ahead"].iloc[-1])
    line = "=" * 80
    print("\n" + line)
    print(f"{f'ПРОГНОЗ {tf.upper()} (каскад) НА {total:.1f} {unit.upper()} ВПЕРЁД':^80}")
    print(line)
    print(f"  hold-out качество: gate MCC={meta['m_gate']['mcc']:.4f} | "
          f"dir MCC={meta['m_dir']['mcc']:.4f} | "
          f"каскад MCC={meta['cascade']['mcc']:.4f}")
    print("-" * 80)
    print(f"{'Дата/Время':<22}{('+'+unit):>7}  {'Сигнал':<7}"
          f"{'P(move)':>9}{'P(UP)':>8}{'Увер.':>9}")
    print("-" * 80)
    for idx, r in res.iterrows():
        pu = f"{r['p_up']:.1%}" if not np.isnan(r["p_up"]) else "—"
        c = f"{r['confidence_pct']:.1f}%" if r["signal"] != 0 else "—"
        print(f"{str(idx):<22}{r['units_ahead']:>6.1f}  "
              f"{r['signal_label']:<7}{r['p_move']:>8.1%}{pu:>8}{c:>9}")
    print(line)
    sig = res[res["signal"] != 0]
    lk = res["last_known_price"].iloc[0]
    print(f"Последняя цена: {lk:.2f} | Сигналов: {len(sig)} | "
          f"UP: {int((res['signal'] == 1).sum())} | "
          f"DOWN: {int((res['signal'] == -1).sum())} | "
          f"FLAT: {int((res['signal'] == 0).sum())}")
    if len(sig) == 0:
        print("  (Каскад не выдал ни одного направленного сигнала.)")
    print(line + "\n")


# 5. ВИЗУАЛИЗАЦИЯ (plotly dark — как в исходных ячейках)

def plot_future(res, tf_bundle, tf, unit, lookback):
    price_full = tf_bundle["price_full"]
    if price_full is None:
        print("  ⚠ нет c_close в бандле — график пропущен")
        return
    hist = price_full.iloc[-lookback:]
    last_price = float(price_full.iloc[-1])
    now_ts = str(price_full.index[-1])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist.index.astype(str), y=hist.to_numpy(), mode="lines",
        name="Цена", line=dict(color=PLOT_COLORS["history"], width=2)))

    anchor_x = [now_ts] + res.index.astype(str).tolist()
    fig.add_trace(go.Scatter(
        x=anchor_x, y=[last_price] * len(anchor_x), mode="lines",
        name="Ориентир",
        line=dict(color=PLOT_COLORS["anchor"], width=1, dash="dot")))

    up = res[res["signal"] == 1]
    if not up.empty:
        fig.add_trace(go.Scatter(
            x=up.index.astype(str), y=[last_price] * len(up),
            mode="markers+text", name="UP",
            marker=dict(color=PLOT_COLORS["up"], size=16,
                        symbol="triangle-up"),
            text=[f"{r['confidence_pct']:.0f}%" for _, r in up.iterrows()],
            textposition="top center",
            textfont=dict(color=PLOT_COLORS["up"], size=10)))

    dn = res[res["signal"] == -1]
    if not dn.empty:
        fig.add_trace(go.Scatter(
            x=dn.index.astype(str), y=[last_price] * len(dn),
            mode="markers+text", name="DOWN",
            marker=dict(color=PLOT_COLORS["down"], size=16,
                        symbol="triangle-down"),
            text=[f"{r['confidence_pct']:.0f}%" for _, r in dn.iterrows()],
            textposition="bottom center",
            textfont=dict(color=PLOT_COLORS["down"], size=10)))

    fig.add_shape(type="line", x0=now_ts, x1=now_ts, y0=0, y1=1,
                  xref="x", yref="paper",
                  line=dict(color=PLOT_COLORS["now"], width=1.5,
                            dash="dash"))
    fig.add_annotation(x=now_ts, y=1, xref="x", yref="paper",
                       text="Сейчас", showarrow=False,
                       font=dict(color=PLOT_COLORS["now"], size=12))
    horizon = float(res["units_ahead"].iloc[-1])
    fig.update_layout(
        title=f"{tf.upper()} каскад-прогноз на {horizon:.1f} {unit} вперёд "
              f"(фичи заморожены — снимок текущих условий)",
        xaxis_title="Дата", yaxis_title="Цена BTC",
        template="plotly_dark", height=600)
    fig.show()


# 6. ЗАПУСК (оба таймфрейма)

if __name__ == "__main__":
    for tf, cfg in TF_CONFIG.items():
        try:
            tfb = load_bundle(tf)
            res, meta = predict_future(
                tf, tfb, cfg["future_bars"], cfg["unit"],
                cfg["unit_div"])
            print_summary(res, tf, cfg["unit"], meta)
            plot_future(res, tfb, tf, cfg["unit"], cfg["lookback"])
        except (FileNotFoundError, KeyError) as e:
            print(f"[{tf.upper()}] пропуск: {e}")